# Sebel Hakim — train the plant-disease model on Kaggle's free GPU

This notebook downloads the public plant-disease datasets, cleans them, trains a large **teacher** model,
distils it into a small **phone** model, tests it on held-out photos and exports it for the Sebel Hakim app.

It runs in **two steps**, so the training gets the whole 12-hour session:

| Run | `MODE` | Accelerator | What it does | Time |
|---|---|---|---|---|
| 1 | `"prepare"` | **None** (CPU; saves your GPU hours) | downloads ~25 datasets, cleans them, saves the photos as this notebook's output | ~4 h |
| 2 | `"train"` | **GPU T4 ×2** | trains teacher + phone model on the saved photos, tests, exports | ~11 h |

**Once, before run 1:** Settings → Phone verification, and join these competitions (open each, click **Join Competition** or
**Late Submission** → **I Understand and Accept**):
https://www.kaggle.com/competitions/cassava-leaf-disease-classification ·
https://www.kaggle.com/competitions/paddy-disease-classification ·
https://www.kaggle.com/competitions/plant-pathology-2021-fgvc8

**Run 1:** set `MODE = "prepare"` below → Session options: Accelerator **None**, Internet **On** → Save Version → Save & Run All.

**Run 2:** create a **new notebook**, import this same file, set `MODE = "train"` → **Add Input → Your Work** → pick the
notebook from run 1 → Session options: **GPU T4 ×2**, Internet **On** → Save Version → Save & Run All.
When it finishes: **Output** tab → download **`sebel_hakim_model.zip`** and send it to Claude.

(`MODE = "all"` still does everything in one run, with less training time.)

In [ ]:
# ---------------- settings ----------------
MODE          = "prepare"                              # "prepare" (run 1, CPU) → "train" (run 2, GPU); or "all"
USE_TEACHER   = True                                   # big teacher → better small model
TEACHER       = "convnextv2_tiny.fcmae_ft_in22k_in1k"
TEACHER_IMG   = 256
TEACHER_MIN   = 300                                    # minutes of training (upper limit; shortened automatically if data prep was slow)
STUDENT       = "efficientnet_b0.ra_in1k"              # the model that goes into the phone app
STUDENT_IMG   = 224
STUDENT_MIN   = 330
EPOCH_SIZE    = 60000                                  # images per epoch (balanced sampling)
RESIZE        = 320                                    # shrink photos once → much faster training
SKIP          = []                                     # dataset ids to skip, e.g. ["plantwild", "plantseg"]
WORKERS       = 4

import os, pathlib, time
START = time.time()                                    # Kaggle stops a run after 12 h; we budget from here
SESSION_MIN = 12 * 60 - 25                             # safety margin
def minutes_left():
    return SESSION_MIN - (time.time() - START) / 60
CODE = pathlib.Path("/kaggle/working/sebel_hakim")
DATA = pathlib.Path("/kaggle/temp/sebel_hakim_data") if pathlib.Path("/kaggle/temp").exists() else pathlib.Path("/tmp/sebel_hakim_data")
RUNS = DATA / "runs"
for p in [CODE / "pipeline", CODE / "kb", CODE / "tests", CODE / "app", CODE / "reports", DATA, RUNS]:
    p.mkdir(parents=True, exist_ok=True)
os.environ["SEBEL_DATA"] = str(DATA)
os.chdir(CODE)
PREPARED = pathlib.Path("/kaggle/working/sebel_hakim_data")   # run-1 output that run 2 reads
assert MODE in ("prepare", "train", "all"), "MODE must be prepare, train or all"
TRAIN = MODE in ("train", "all")
print("MODE:", MODE, "| code:", CODE, "| data:", DATA)

In [ ]:
# ---------------- check GPU, internet and attached inputs ----------------
import subprocess, urllib.request, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — turn on GPU in Session options!")
try:
    urllib.request.urlopen("https://github.com", timeout=10); print("Internet: on")
except Exception:
    print("Internet: OFF — turn it on in Session options (needs phone verification)")
print("Attached inputs:", sorted(p.name for p in pathlib.Path("/kaggle/input").glob("*")) if pathlib.Path("/kaggle/input").exists() else "none")
subprocess.run("pip install -q onnx onnxscript onnxruntime huggingface_hub kagglehub", shell=True)
import timm; print("timm", timm.__version__, "| torch", torch.__version__)

## Code and knowledge base
The cells below write the Sebel Hakim pipeline and disease knowledge base to disk. You don't need to read them.

In [ ]:
%%writefile pipeline/build_kb.py
"""Compile kb/*.yaml into app/kb.json (what the phone app reads).

  python pipeline/build_kb.py
"""
import json

from common import KB_DIR, ROOT, load_diseases, load_yaml

TEXT_FIELDS = ["symptoms", "spread", "cultural", "biological", "chemical", "prevention", "notes"]


def main():
    crops = load_yaml(KB_DIR / "crops.yaml")
    diseases = []
    for d in load_diseases():
        e = {k: d[k] for k in ("id", "crop", "type") if k in d}
        e["severity"] = d.get("severity", "none" if d["type"] == "healthy" else "medium")
        e["name"] = d.get("name", {})
        if d.get("scientific"):
            e["scientific"] = d["scientific"]
        for k in TEXT_FIELDS:
            if d.get(k):
                e[k] = d[k]
        diseases.append(e)
    out = ROOT / "app" / "kb.json"
    out.parent.mkdir(exist_ok=True)
    out.write_text(json.dumps({"crops": crops["crops"], "groups": crops["groups"], "diseases": diseases},
                              ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
    print(f"{len(diseases)} diseases, {len(crops['crops'])} crops → {out} ({out.stat().st_size / 1024:.0f} KB)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile pipeline/common.py
"""Shared helpers: paths, knowledge-base loading, label matching."""
from __future__ import annotations

import os
import re
from dataclasses import dataclass, field
from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parent.parent          # repo root
KB_DIR = ROOT / "kb"
PIPE_DIR = ROOT / "pipeline"
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
# where downloaded/prepared data lives (big); override with SEBEL_DATA, e.g. on Kaggle
DATA_ROOT = Path(os.environ.get("SEBEL_DATA", ROOT / "data"))


def walk_files(base: Path):
    """All files under base, following symlinks (Kaggle inputs are linked in)."""
    for d, dirs, files in os.walk(base, followlinks=True):
        dirs[:] = [x for x in dirs if not x.startswith(".")]
        for f in files:
            yield Path(d) / f


def norm(s: str) -> str:
    """Lowercase, turn every run of non-alphanumerics into one space."""
    s = str(s).lower().replace("(", " ").replace(")", " ")
    return re.sub(r"[^0-9a-zሀ-፿]+", " ", s).strip()


def load_yaml(p: Path):
    with open(p, encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_crops() -> dict:
    data = load_yaml(KB_DIR / "crops.yaml")
    return {c["id"]: c for c in data["crops"]}


def load_diseases() -> list[dict]:
    out = []
    for p in sorted(KB_DIR.glob("*.yaml")):
        if p.name == "crops.yaml":
            continue
        out.extend(load_yaml(p).get("diseases", []))
    ids = [d["id"] for d in out]
    dupes = {i for i in ids if ids.count(i) > 1}
    if dupes:
        raise ValueError(f"duplicate disease ids in kb/: {sorted(dupes)}")
    crops = load_crops()
    for d in out:
        if d["crop"] not in crops:
            raise ValueError(f"{d['id']}: unknown crop '{d['crop']}' (add it to kb/crops.yaml)")
        if not d["id"].startswith(d["crop"] + "."):
            raise ValueError(f"{d['id']}: id must start with '{d['crop']}.'")
    return out


def load_datasets() -> list[dict]:
    return load_yaml(PIPE_DIR / "datasets.yaml")["datasets"]


# Words that name a crop inside dataset folder names (e.g. "Corn_(maize)___Common_rust_").
CROP_WORDS = {
    "maize": ["maize", "corn"], "wheat": ["wheat"], "rice": ["rice", "paddy"],
    "sorghum": ["sorghum"], "bean": ["bean", "beans"], "soybean": ["soybean", "soyabean", "soya"],
    "groundnut": ["groundnut", "peanut"], "cassava": ["cassava"], "potato": ["potato"],
    "tomato": ["tomato"], "pepper": ["pepper", "bell pepper", "capsicum"], "squash": ["squash", "pumpkin"],
    "banana": ["banana"], "mango": ["mango"], "citrus": ["citrus", "orange", "lemon"],
    "apple": ["apple"], "grape": ["grape"], "peach": ["peach"], "cherry": ["cherry"],
    "strawberry": ["strawberry"], "blueberry": ["blueberry"], "raspberry": ["raspberry"],
    "coffee": ["coffee"], "guava": ["guava"], "papaya": ["papaya", "pawpaw"],
}


def detect_crop(label: str) -> str | None:
    """Return the crop named in a folder label, preferring the longest match."""
    n = " " + norm(label) + " "
    best, best_len = None, 0
    for crop, words in CROP_WORDS.items():
        for w in words:
            if f" {w} " in n and len(w) > best_len:
                # "soybean" contains "bean": longest wins; "bell pepper" beats "pepper"
                best, best_len = crop, len(w)
    return best


@dataclass
class Matcher:
    """Maps raw dataset labels onto knowledge-base disease ids."""
    diseases: list[dict]
    by_syn: dict = field(default_factory=dict)       # norm(synonym) -> set(ids)

    def __post_init__(self):
        self.ids = {d["id"] for d in self.diseases}
        for d in self.diseases:
            names = [d["id"], d["id"].replace(".", " "), d.get("name", {}).get("en", "")]
            for s in names + list(d.get("synonyms", []) or []):
                if s:
                    self.by_syn.setdefault(norm(s), set()).add(d["id"])

    def match(self, label: str, crop: str | None = None, overrides: dict | None = None) -> str | None:
        r = self._match(label, crop, overrides)
        if r is None and crop:
            # numbered folders from split archives: "mln4", "als_2", "healthy10", "msv 3"
            base = re.sub(r"[\s_\-]*\d+$", "", str(label)).strip()
            if base and base != str(label).strip():
                r = self._match(base, crop, overrides)
        return r

    def _match(self, label: str, crop: str | None = None, overrides: dict | None = None) -> str | None:
        """Return a disease id or None.

        Order: explicit override → exact synonym (filtered by crop) → synonym after
        removing the crop words → longest synonym contained in the label.
        """
        if overrides:
            for k, v in overrides.items():
                if norm(k) == norm(label):
                    return v or None          # empty override = deliberately skip
        crop = crop or detect_crop(label)
        n = norm(label)

        def pick(ids: set[str]) -> str | None:
            if crop:
                ids = {i for i in ids if i.startswith(crop + ".")}
            return next(iter(ids)) if len(ids) == 1 else None

        if n.isdigit() and not crop:
            return None                    # numeric class ids only mean something inside one dataset
        if n in self.by_syn and (r := pick(self.by_syn[n])):
            return r
        if crop:
            stripped = n
            for w in sorted(CROP_WORDS.get(crop, []), key=len, reverse=True):
                stripped = re.sub(rf"\b{re.escape(w)}\b", " ", stripped)
            stripped = re.sub(r"\b(leaf|leaves|plant|disease|images?)\b", " ", stripped)
            stripped = re.sub(r"\s+", " ", stripped).strip()
            if stripped in self.by_syn and (r := pick(self.by_syn[stripped])):
                return r
            if stripped in ("", "healthy", "normal", "healthy healthy", "fresh", "good"):
                # "Strawberry leaf", "Healthy", "normal" → that crop's healthy class
                return f"{crop}.healthy" if f"{crop}.healthy" in self.ids else None
        # longest synonym fully contained in the label (only when the crop is known)
        if crop:
            best, best_len = None, 0
            for syn, ids in self.by_syn.items():
                if len(syn) >= 4 and f" {syn} " in f" {n} ":
                    r = pick(ids)
                    if r and len(syn) > best_len:
                        best, best_len = r, len(syn)
            return best
        return None

In [ ]:
%%writefile pipeline/download.py
"""Download the public datasets listed in datasets.yaml into data/raw/<id>/.

Usage:
    python pipeline/download.py                 # everything that can be fetched automatically
    python pipeline/download.py --only plantvillage cassava_kaggle
    python pipeline/download.py --list          # show datasets and their status

Already-downloaded datasets are skipped (a `.done` marker is written).
Archives (.zip/.tar/.tar.gz/.7z) are extracted in place.
Datasets with `host: manual` print instructions instead.
"""
from __future__ import annotations

import argparse
import json
import os
import shutil
import subprocess
import sys
import tarfile
import urllib.parse
import urllib.request
import zipfile
from pathlib import Path

from common import DATA_ROOT, IMG_EXT, ROOT, load_datasets, walk_files

RAW = DATA_ROOT / "raw"
KAGGLE_INPUT = Path(os.environ.get("KAGGLE_INPUT", "/kaggle/input"))
UA = {"User-Agent": "sebel-hakim-dataset-downloader/1.0"}


def http_json(url: str):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.load(r)


def http_download(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=600) as r, open(tmp, "wb") as f:
        total = int(r.headers.get("Content-Length") or 0)
        done = 0
        while chunk := r.read(1 << 20):
            f.write(chunk)
            done += len(chunk)
            if total:
                print(f"\r    {dest.name}: {done / total:6.1%} of {total / 1e6:,.0f} MB", end="", flush=True)
    print()
    tmp.rename(dest)


def extract_all(folder: Path):
    """Extract every archive under folder (repeat once for archives inside archives)."""
    for _ in range(3):
        for a in list(folder.rglob("*")):
            name = a.name.lower()
            try:
                if name.endswith(".zip"):
                    with zipfile.ZipFile(a) as z:
                        z.extractall(a.parent / a.stem)
                elif name.endswith((".tar", ".tar.gz", ".tgz")):
                    with tarfile.open(a) as t:
                        t.extractall(a.parent / a.name.split(".tar")[0], filter="data")
                elif name.endswith((".7z", ".rar")):
                    dest = a.parent / a.stem
                    dest.mkdir(exist_ok=True)
                    for cmd in (["7z", "x", "-y", f"-o{dest}", str(a)], ["unrar", "x", "-o+", str(a), str(dest) + "/"],
                                ["bsdtar", "-xf", str(a), "-C", str(dest)]):
                        if shutil.which(cmd[0]) and subprocess.run(cmd, capture_output=True).returncode == 0:
                            break
                    else:
                        raise RuntimeError("no working 7z / unrar / bsdtar for this archive")
                else:
                    continue
                a.unlink()
                print(f"    extracted {a.name}")
            except Exception as e:  # keep going; prepare.py reports what is usable
                print(f"    ! could not extract {a.name}: {e}")


# ---------------------------------------------------------------- hosts
def get_github(d, out: Path):
    url = f"https://github.com/{d['ref']}.git"
    if d.get("subdir"):
        # sparse, blob-less clone: only the folder we need
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", url, str(out)], check=True)
        subprocess.run(["git", "-C", str(out), "sparse-checkout", "set", d["subdir"]], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", url, str(out)], check=True)
    shutil.rmtree(out / ".git", ignore_errors=True)


def get_zenodo(d, out: Path):
    rec = http_json(f"https://zenodo.org/api/records/{d['ref']}")
    for f in rec.get("files", []):
        key = f.get("key") or f.get("filename")
        link = f["links"].get("self") or f["links"].get("download")
        http_download(link, out / key)


def get_mendeley(d, out: Path):
    """Try the listed version first, then other versions (datasets get new versions over time)."""
    errors = []
    for v in [d.get("version", 1)] + [x for x in (5, 4, 3, 2, 1) if x != d.get("version", 1)]:
        try:
            return _get_mendeley({**d, "version": v}, out)
        except Exception as e:
            errors.append(f"v{v}: {str(e)[:150]}")
            for f in out.iterdir():
                if f.is_file():
                    f.unlink()
    raise RuntimeError(" || ".join(errors)[:600])


def _get_mendeley(d, out: Path):
    """Mendeley Data has changed its public endpoints over time, so try several in turn."""
    ver, ref = d.get("version", 1), d["ref"]
    errors = []
    # 1) whole-dataset zip
    for url in (f"https://data.mendeley.com/public-api/zip/{ref}/download/{ver}",
                f"https://data.mendeley.com/public-api/zip/{ref}/download/{ver}?format=zip"):
        try:
            http_download(url, out / f"{ref}_v{ver}.zip")
            if (out / f"{ref}_v{ver}.zip").stat().st_size > 10_000:
                return
        except Exception as e:
            errors.append(f"zip: {e}")
    # 2) file-by-file listing (several API shapes)
    for base in (f"https://data.mendeley.com/public-api/datasets/{ref}",
                 f"https://data.mendeley.com/api/datasets/{ref}"):
        try:
            meta = http_json(f"{base}?version={ver}")
            files = meta.get("files") or http_json(f"{base}/files?folder_id=root&version={ver}")
            n = 0
            for f in files:
                cd = f.get("content_details") or {}
                link = cd.get("download_url") or f.get("download_url")
                if link:
                    http_download(link, out / f["filename"]); n += 1
            if n:
                return
            errors.append(f"{base}: no file links")
        except Exception as e:
            errors.append(f"{base}: {e}")
    # 3) a Kaggle mirror, if one is listed
    if d.get("kaggle_mirror"):
        try:
            get_kaggle({**d, "ref": d["kaggle_mirror"]}, out, comp=False)
            return
        except Exception as e:
            errors.append(f"kaggle mirror: {e}")
    raise RuntimeError("Mendeley download failed (" + " | ".join(errors)[:400] +
                       f") — download manually from https://data.mendeley.com/datasets/{ref}/{ver}")


def get_dataverse(d, out: Path):
    pid = urllib.parse.quote(d["ref"], safe="")
    http_download(f"https://dataverse.harvard.edu/api/access/dataset/:persistentId/?persistentId={pid}",
                  out / "dataset.zip")


def find_kaggle_input(slug: str) -> Path | None:
    """A dataset/competition attached with "Add Input" in a Kaggle notebook."""
    if not KAGGLE_INPUT.is_dir():
        return None
    for depth1 in KAGGLE_INPUT.iterdir():
        if depth1.name == slug:
            return depth1
        if depth1.is_dir():                      # newer layout: /kaggle/input/<kind>/<owner>/<slug>
            for p in depth1.rglob(slug):
                if p.is_dir():
                    return p
    return None


def get_kaggle(d, out: Path, comp: bool):
    src = find_kaggle_input(d["ref"].split("/")[-1])
    if src is not None:
        (out / "input").symlink_to(src, target_is_directory=True)
        print(f"    using attached Kaggle input {src}")
        return
    # inside a Kaggle notebook: kagglehub fetches it with the notebook owner's access
    # (the owner must have joined the competition / accepted its rules once on kaggle.com)
    try:
        import kagglehub
        path = (kagglehub.competition_download(d["ref"]) if comp else kagglehub.dataset_download(d["ref"]))
        (out / "input").symlink_to(Path(path), target_is_directory=True)
        print(f"    fetched with kagglehub → {path}")
        return
    except ImportError:
        pass
    except Exception as e:
        hint = (f" — open https://www.kaggle.com/competitions/{d['ref']} and click Join / Late Submission → "
                "I Understand and Accept, then run again") if comp else ""
        raise RuntimeError(f"kagglehub could not get {d['ref']}: {e}{hint}") from e
    if not shutil.which("kaggle"):
        raise RuntimeError("install the Kaggle CLI (pip install kaggle) and put your token in ~/.kaggle/kaggle.json")
    cmd = (["kaggle", "competitions", "download", "-c", d["ref"]] if comp
           else ["kaggle", "datasets", "download", "-d", d["ref"]])
    subprocess.run(cmd + ["-p", str(out)], check=True)


def get_hf(d, out: Path):
    try:
        from huggingface_hub import snapshot_download
    except ImportError as e:
        raise RuntimeError("pip install huggingface_hub") from e
    snapshot_download(repo_id=d["ref"], repo_type="dataset", local_dir=str(out))
    # parquet-based HF datasets are turned into class folders by prepare.py


HOSTS = {
    "github": get_github, "zenodo": get_zenodo, "mendeley": get_mendeley,
    "dataverse": get_dataverse, "hf": get_hf,
    "kaggle": lambda d, o: get_kaggle(d, o, comp=False),
    "kaggle_comp": lambda d, o: get_kaggle(d, o, comp=True),
}


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--only", nargs="*", help="dataset ids to download")
    ap.add_argument("--skip", nargs="*", default=[], help="dataset ids to skip")
    ap.add_argument("--list", action="store_true")
    a = ap.parse_args()

    ds = load_datasets()
    if a.list:
        for d in ds:
            status = "done" if (RAW / d["id"] / ".done").exists() else "-"
            print(f"{d['id']:24} {d['host']:12} {d.get('field',''):6} {str(d.get('commercial')):8} {status}")
        return

    failed, manual, report = [], [], []
    for d in ds:
        if (a.only and d["id"] not in a.only) or d["id"] in a.skip:
            continue
        out = RAW / d["id"]
        if (out / ".done").exists():
            print(f"[skip] {d['id']} already downloaded")
            report.append((d["id"], "ok (earlier)", ""))
            continue
        if out.exists() and not (out / ".done").exists():
            shutil.rmtree(out, ignore_errors=True)   # half-finished earlier attempt
        if d["host"] == "manual":
            manual.append(d)
            report.append((d["id"], "manual", d.get("url", "")))
            continue
        print(f"[get ] {d['id']} ({d['host']}: {d.get('ref')})")
        out.mkdir(parents=True, exist_ok=True)
        try:
            HOSTS[d["host"]](d, out)
            if not (out / "input").is_symlink():  # never touch read-only Kaggle inputs
                extract_all(out)
            n = sum(1 for p in walk_files(out) if p.suffix.lower() in IMG_EXT or p.suffix == ".parquet")
            if n == 0:
                from collections import Counter
                kinds = Counter((p.suffix.lower() or "(none)") for p in walk_files(out))
                raise RuntimeError("downloaded, but no image files were found inside; file types: " +
                                   (", ".join(f"{k} ×{v}" for k, v in kinds.most_common(8)) or "none"))
            (out / ".done").write_text("ok\n")
            report.append((d["id"], "ok", f"{n:,} files"))
        except Exception as e:
            print(f"  ! {d['id']} failed: {e}")
            failed.append(d["id"])
            report.append((d["id"], "FAILED", str(e).replace("|", "/").replace("\n", " ")[:300]))
            shutil.rmtree(out, ignore_errors=True)

    rep = ROOT / "reports" / "download_report.md"
    rep.parent.mkdir(exist_ok=True)
    rep.write_text("# Download report\n\n| dataset | status | details |\n|---|---|---|\n" +
                   "".join(f"| {i} | {s} | {x} |\n" for i, s, x in report), encoding="utf-8")
    print(f"\n→ {rep}")
    if manual:
        print("\nManual downloads (put the files in data/raw/<id>/ and create an empty .done file):")
        for d in manual:
            if not (RAW / d["id"] / ".done").exists():
                print(f"  - {d['id']}: {d.get('url')}\n      {d.get('note', '')}")
    if failed:
        print(f"\nFailed: {', '.join(failed)}  (re-run later, or download manually)")
        sys.exit(1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile pipeline/evaluate.py
"""Evaluate a checkpoint on the held-out test split.

Reports accuracy and macro-F1 overall, on field photos only, per dataset, per crop and per class,
plus the most common confusions and how often a "not sure" warning would be shown.

  python pipeline/evaluate.py runs/student/best.pt            # PyTorch checkpoint
  python pipeline/evaluate.py app/model/model.onnx            # exported ONNX model (what the app runs)
"""
from __future__ import annotations

import argparse
import json
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from common import DATA_ROOT, ROOT
from train import ImgDS, eval_tf, macro_f1, read_manifest


def load_runner(path: Path):
    """Return (classes, img, temperature, fn(batch_tensor)->logits)."""
    if path.suffix == ".onnx":
        import onnxruntime as ort
        meta = json.loads((path.parent / "labels.json").read_text())
        sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
        name = sess.get_inputs()[0].name
        return ([c["id"] for c in meta["classes"]], meta["img"], meta["temperature"],
                lambda x: torch.from_numpy(sess.run(None, {name: x.numpy()})[0]))
    import timm
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m = timm.create_model(ck["model"], pretrained=False, num_classes=len(ck["classes"]))
    m.load_state_dict(ck["state_dict"])
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    m.eval().to(dev)

    @torch.no_grad()
    def run(x):
        return m(x.to(dev)).float().cpu()
    return ck["classes"], ck["img"], ck["temperature"], run


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model")
    ap.add_argument("--manifest", default=str(DATA_ROOT / "manifest.csv"))
    ap.add_argument("--split", default="test")
    ap.add_argument("--threshold", type=float, default=0.5, help="'not sure' below this confidence")
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--out", default="")
    ap.add_argument("--max-images", type=int, default=0, help="evaluate a random subset (quick checks)")
    a = ap.parse_args()

    path = Path(a.model)
    classes, img, temp, run = load_runner(path)
    rows = [r for r in read_manifest(Path(a.manifest)) if r["split"] == a.split and r["label"] in classes]
    if a.max_images and len(rows) > a.max_images:
        import random
        rows = random.Random(0).sample(rows, a.max_images)
    dl = DataLoader(ImgDS(rows, classes, eval_tf(img)), batch_size=a.bs, num_workers=a.workers)
    probs, ys = [], []
    for x, y in dl:
        probs.append(torch.softmax(run(x) / temp, -1))
        ys.append(y)
    P, y = torch.cat(probs).numpy(), torch.cat(ys).numpy()
    p = P.argmax(1)
    conf = P.max(1)
    top3 = (np.argsort(-P, 1)[:, :3] == y[:, None]).any(1)
    n = len(classes)

    def block(mask):
        if not mask.any():
            return None
        return {"images": int(mask.sum()), "acc": float((p[mask] == y[mask]).mean()),
                "top3": float(top3[mask].mean()), "macro_f1": macro_f1(y[mask], p[mask], n)}

    field = np.array([r["field"] in ("field", "mixed") for r in rows])
    crop_of = np.array([c.split(".")[0] for c in classes])
    res = {"model": str(path), "split": a.split, "overall": block(np.ones_like(y, bool)),
           "field_only": block(field), "lab_only": block(~field),
           "crop_correct": float((crop_of[p] == crop_of[y]).mean()),
           "not_sure_rate": float((conf < a.threshold).mean()),
           "acc_when_sure": float((p[conf >= a.threshold] == y[conf >= a.threshold]).mean()) if (conf >= a.threshold).any() else None,
           "per_dataset": {}, "per_crop": {}, "per_class": {}, "top_confusions": []}
    ds = np.array([r["dataset"] for r in rows])
    for d in sorted(set(ds)):
        res["per_dataset"][d] = block(ds == d)
    ycrop = crop_of[y]
    for c in sorted(set(ycrop)):
        res["per_crop"][c] = block(ycrop == c)
    for i, c in enumerate(classes):
        m = y == i
        if m.any():
            tp = int(((p == i) & m).sum())
            res["per_class"][c] = {"images": int(m.sum()), "recall": tp / int(m.sum()),
                                   "precision": tp / max(int((p == i).sum()), 1)}
    conf_pairs = Counter((classes[a_], classes[b_]) for a_, b_ in zip(y, p) if a_ != b_)
    res["top_confusions"] = [{"true": t, "predicted": q, "count": k} for (t, q), k in conf_pairs.most_common(25)]

    out = Path(a.out) if a.out else ROOT / "reports" / f"eval_{path.parent.name}_{a.split}.json"
    out.parent.mkdir(exist_ok=True)
    out.write_text(json.dumps(res, indent=1))

    def fmt(b):
        return "n/a" if not b else f"acc {b['acc']:.3f} · top-3 {b['top3']:.3f} · macro-F1 {b['macro_f1']:.3f} ({b['images']:,} imgs)"
    print(f"overall     {fmt(res['overall'])}")
    print(f"field only  {fmt(res['field_only'])}")
    print(f"lab only    {fmt(res['lab_only'])}")
    print(f"crop right  {res['crop_correct']:.3f}   'not sure' shown {res['not_sure_rate']:.1%}   "
          f"accuracy when sure {res['acc_when_sure'] if res['acc_when_sure'] is None else round(res['acc_when_sure'], 3)}")
    print("per crop:")
    for c, b in res["per_crop"].items():
        print(f"  {c:11} {fmt(b)}")
    print(f"→ {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile pipeline/export.py
"""Export a trained checkpoint for the app.

Writes app/model/model.onnx (+ model.int8.onnx when --quantize) and app/model/labels.json,
then checks the ONNX output matches PyTorch.

  python pipeline/export.py runs/student/best.pt [--quantize]
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import timm
import torch

from common import DATA_ROOT, ROOT, load_diseases


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("ckpt")
    ap.add_argument("--out", default=str(ROOT / "app" / "model"))
    ap.add_argument("--quantize", action="store_true", help="also write an int8 model (about 4x smaller)")
    ap.add_argument("--calib", type=int, default=300, help="calibration images for int8")
    a = ap.parse_args()

    ck = torch.load(a.ckpt, map_location="cpu", weights_only=False)
    model = timm.create_model(ck["model"], pretrained=False, num_classes=len(ck["classes"]), exportable=True)
    model.load_state_dict(ck["state_dict"])
    model.eval()
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    x = torch.randn(1, 3, ck["img"], ck["img"])
    onnx_path = out / "model.onnx"
    kw = dict(input_names=["image"], output_names=["logits"],
              dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17)
    try:
        torch.onnx.export(model, x, str(onnx_path), dynamo=False, **kw)
    except TypeError:                      # torch < 2.5 has no `dynamo` argument
        torch.onnx.export(model, x, str(onnx_path), **kw)

    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    with torch.no_grad():
        ref = model(x).numpy()
    got = sess.run(None, {"image": x.numpy()})[0]
    diff = float(np.abs(ref - got).max())
    print(f"onnx check: max |diff| = {diff:.2e}")
    if diff > 1e-2:
        raise SystemExit("ONNX output differs from PyTorch; not exporting")

    dmap = {d["id"]: d for d in load_diseases()}
    labels = {
        "model": ck["model"], "img": ck["img"], "mean": list(ck["mean"]), "std": list(ck["std"]),
        "temperature": ck["temperature"], "epoch": ck.get("epoch"), "val": ck.get("val"),
        "classes": [{"id": c, "crop": dmap[c]["crop"], "type": dmap[c]["type"]} for c in ck["classes"]],
        "files": {"fp32": "model.onnx"},
    }

    if a.quantize:
        from onnxruntime.quantization import CalibrationDataReader, QuantFormat, QuantType, quantize_static
        from onnxruntime.quantization.shape_inference import quant_pre_process
        import csv
        from PIL import Image
        from train import eval_tf
        from train import read_manifest
        rows = [r for r in read_manifest(DATA_ROOT / "manifest.csv") if r["split"] == "train"]
        rng = np.random.default_rng(0)
        pick = [rows[i] for i in rng.choice(len(rows), min(a.calib, len(rows)), replace=False)]
        tf = eval_tf(ck["img"])

        class Reader(CalibrationDataReader):
            def __init__(self):
                self.it = iter(pick)

            def get_next(self):
                r = next(self.it, None)
                if r is None:
                    return None
                with Image.open(r["path"]) as im:
                    return {"image": tf(im.convert("RGB")).unsqueeze(0).numpy()}

        pre = out / "model.pre.onnx"
        quant_pre_process(str(onnx_path), str(pre))
        q = out / "model.int8.onnx"
        quantize_static(str(pre), str(q), Reader(), quant_format=QuantFormat.QDQ,
                        activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8, per_channel=True)
        pre.unlink()
        labels["files"]["int8"] = "model.int8.onnx"
        print(f"int8 model: {q.stat().st_size / 1e6:.1f} MB (check its accuracy with evaluate.py before using it)")

    (out / "labels.json").write_text(json.dumps(labels, indent=1))
    print(f"fp32 model: {onnx_path.stat().st_size / 1e6:.1f} MB, {len(ck['classes'])} classes → {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile pipeline/prepare.py
"""Build one clean, labelled manifest from all downloaded datasets.

Steps
  1. Read every dataset in data/raw/ (class folders, CSV labels or Hugging Face parquet).
  2. Map each raw label to a knowledge-base disease id (kb/*.yaml synonyms,
     plus per-dataset `map:` overrides in datasets.yaml).
  3. Drop unreadable images and near-duplicates (perceptual hash), so the same
     photo can never sit in both train and test.
  4. Split train / val / test per dataset and class.
  5. Write data/manifest.csv, data/classes.json and reports/prepare_report.md.

Usage
  python pipeline/prepare.py [--commercial-only] [--min-per-class 20] [--workers 16]
"""
from __future__ import annotations

import argparse
import csv
import functools
import hashlib
import json
import random
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

from common import DATA_ROOT, IMG_EXT, ROOT, Matcher, detect_crop, load_datasets, load_diseases, norm, walk_files

RAW = DATA_ROOT / "raw"
DATA = DATA_ROOT
REPORTS = ROOT / "reports"
GENERIC_DIRS = {"train", "test", "val", "valid", "validation", "images", "image", "img", "data",
                "dataset", "color", "raw", "train_images", "test_images", "original", "augmented"}


# ---------------------------------------------------------------- collecting
def export_parquet(folder: Path) -> None:
    """Turn Hugging Face parquet files into <folder>/_hf/<label>/<n>.jpg once."""
    out = folder / "_hf"
    files = [str(p) for p in walk_files(folder) if p.suffix == ".parquet"]
    if out.exists() or not files:
        return
    from datasets import load_dataset
    ds = load_dataset("parquet", data_files=files, split="train")
    img_col = next(c for c, f in ds.features.items() if f.__class__.__name__ == "Image")
    lab_col = next((c for c, f in ds.features.items() if f.__class__.__name__ == "ClassLabel"), None)
    if lab_col is None:
        lab_col = next(c for c in ds.column_names if c.lower() in {"label", "labels", "class", "disease"})
        names = None
    else:
        names = ds.features[lab_col].names
    for i, row in enumerate(ds):
        lab = names[row[lab_col]] if names else str(row[lab_col])
        p = out / lab / f"{i:07d}.jpg"
        p.parent.mkdir(parents=True, exist_ok=True)
        row[img_col].convert("RGB").save(p, quality=92)


def folder_label(path: Path, base: Path, depth: int | None) -> str:
    parts = path.relative_to(base).parts[:-1]
    if depth is not None:
        return parts[depth] if len(parts) > depth else ""
    for p in reversed(parts):          # nearest non-generic parent folder
        if norm(p) not in {norm(g) for g in GENERIC_DIRS}:
            return p
    return ""


def collect(d: dict) -> list[tuple[Path, str]]:
    base = RAW / d["id"]
    if d["host"] == "hf":
        export_parquet(base)
    items: list[tuple[Path, str]] = []
    if d.get("label") == "csv":
        spec = d["csv"]
        csv_path = next((p for p in walk_files(base) if p.name == spec["file"]), None)
        img_dir = next((p.parent for p in walk_files(base)
                        if p.parent.name == spec["image_dir"] and p.suffix.lower() in IMG_EXT), None)
        if not csv_path or not img_dir:
            print(f"  ! {d['id']}: {spec['file']} or {spec['image_dir']} not found")
            return []
        sep = spec.get("multi_label_sep")
        with open(csv_path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                lab = str(row[spec["label_col"]])
                if sep and len(lab.split(sep)) > 1:
                    continue                       # skip multi-disease images
                items.append((img_dir / row[spec["image_col"]], lab))
        return items
    scan = base
    if d.get("label_dir"):
        scan = next((p.parent.parent for p in walk_files(base)
                     if p.parent.parent.name == d["label_dir"] and p.suffix.lower() in IMG_EXT), base)
    excl = [norm(x) for x in d.get("exclude", []) or []]
    for p in walk_files(scan):
        if p.suffix.lower() not in IMG_EXT:
            continue
        rel = p.relative_to(scan)
        if excl and any(x in norm(" ".join(rel.parts[:-1])) for x in excl):
            continue                            # e.g. augmented copies, unlabelled "disease fruit"
        if d.get("label_join"):                 # "Mango/Healthy Fruit" → "Mango Healthy Fruit"
            parts = [x for x in rel.parts[:-1] if norm(x) not in {norm(g) for g in GENERIC_DIRS}]
            items.append((p, " ".join(parts)))
        else:
            items.append((p, folder_label(p, scan, d.get("label_depth"))))
    return items


# ---------------------------------------------------------------- hashing
def hash_one(path: str, cache: str = "", size: int = 0):
    """Difference hash (64 bit) of a 9x8 grayscale thumbnail: identical for re-saved / resized copies.

    With `cache`, also writes a copy whose short side is `size` px, so training decodes small JPEGs.
    Returns (original path, hash or None, path to use for training).
    """
    try:
        import numpy as np
        from PIL import Image, ImageOps
        with Image.open(path) as im:
            if cache:
                im.draft("RGB", (size * 2, size * 2))
                im = ImageOps.exif_transpose(im).convert("RGB")
                if min(im.size) < 32:
                    return path, None, path
                if min(im.size) > size:
                    s = size / min(im.size)
                    im = im.resize((round(im.width * s), round(im.height * s)), Image.Resampling.BILINEAR)
                dst = Path(cache) / (hashlib.sha1(path.encode()).hexdigest()[:20] + ".jpg")
                im.save(dst, quality=90)
                use = str(dst)
            else:
                im.draft("L", (64, 64))             # fast JPEG decode at reduced size
                use = path
            g = im.convert("L")
            if min(g.size) < 32:
                return path, None, path
            a = np.asarray(g.resize((9, 8), Image.Resampling.LANCZOS), dtype=np.int16)
        bits = (a[:, 1:] > a[:, :-1]).flatten()
        return path, f"{int(''.join('1' if b else '0' for b in bits), 2):016x}", use
    except Exception:
        return path, None, path


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--commercial-only", action="store_true", help="drop datasets with non-commercial licences")
    ap.add_argument("--min-per-class", type=int, default=20)
    ap.add_argument("--val", type=float, default=0.10)
    ap.add_argument("--test", type=float, default=0.10)
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--no-dedup", action="store_true")
    ap.add_argument("--max-per-class", type=int, default=6000,
                    help="cap images per class per dataset (random sample) so huge sets don't swamp the rest; 0 = no cap")
    ap.add_argument("--resize", type=int, default=0,
                    help="write copies with this short side (e.g. 320) for much faster training")
    a = ap.parse_args()

    diseases = load_diseases()
    kb_ids = {d["id"] for d in diseases}
    m = Matcher(diseases)
    rows, unmapped = [], defaultdict(Counter)

    for d in load_datasets():
        if not (RAW / d["id"] / ".done").exists():
            continue
        if a.commercial_only and d.get("commercial") is not True:
            print(f"[drop] {d['id']} (licence: {d.get('licence')})")
            continue
        items = collect(d)
        mapped = 0
        cache: dict[str, str | None] = {}
        for path, raw in items:
            if raw not in cache:
                cache[raw] = m.match(raw, crop=d.get("crop"), overrides=d.get("map"))
                # a plant we don't cover at all (no known crop word in the label) → "other plant"
                if (cache[raw] is None and d.get("other_plants") and not d.get("crop")
                        and raw and not raw.isdigit() and detect_crop(raw) is None
                        and norm(raw) not in m.by_syn       # bare "healthy", "rust" … stay unmapped
                        and "other.unknown_plant" in kb_ids):
                    cache[raw] = "other.unknown_plant"
            lab = cache[raw]
            if lab is None:
                unmapped[d["id"]][raw] += 1
                continue
            rows.append({"path": str(path), "label": lab, "dataset": d["id"],
                         "field": d.get("field", "lab"), "weight": d.get("weight", 1.0), "raw": raw})
            mapped += 1
        cap = d.get("max_per_class", a.max_per_class)
        if cap:
            mine = [r for r in rows if r["dataset"] == d["id"]]
            by = defaultdict(list)
            for r in mine:
                by[r["label"]].append(r)
            keep = []
            rnd = random.Random(a.seed)
            for lab, g in by.items():
                rnd.shuffle(g)
                keep += g[:cap]
            if len(keep) < len(mine):
                print(f"       capped {d['id']} to {cap:,} per class: {len(mine):,} → {len(keep):,}")
            rows = [r for r in rows if r["dataset"] != d["id"]] + keep
        print(f"[read] {d['id']:24} {len(items):7} images, {mapped:7} mapped, "
              f"{len(cache)} raw classes → {len({v for v in cache.values() if v})} kb classes")

    if not rows:
        raise SystemExit("No images found. Run pipeline/download.py first.")

    # ---- readability + duplicates
    print(f"[hash] {len(rows)} images with {a.workers} workers …")
    cache = ""
    if a.resize:
        cache = str(DATA / f"img{a.resize}")
        Path(cache).mkdir(parents=True, exist_ok=True)
        print(f"       also writing {a.resize}px copies to {cache}")
    fn = functools.partial(hash_one, cache=cache, size=a.resize)
    res = {}
    with ProcessPoolExecutor(a.workers) as ex:
        for i, (pth, h, use) in enumerate(ex.map(fn, [r["path"] for r in rows], chunksize=64)):
            res[pth] = (h, use)
            if i % 10000 == 0 and i:
                print(f"       {i:,}/{len(rows):,}", flush=True)
    bad = sum(1 for r in rows if res[r["path"]][0] is None)
    rows = [r | {"hash": res[r["path"]][0], "path": res[r["path"]][1], "source": r["path"]}
            for r in rows if res[r["path"]][0]]
    if cache:                                   # store "img320/xxx.jpg" relative to the manifest folder
        for r in rows:
            if r["path"].startswith(cache + "/"):
                r["path"] = r["path"][len(str(DATA)) + 1:]
    dup_removed, conflict_removed = 0, 0
    if not a.no_dedup:
        by_hash = defaultdict(list)
        for r in rows:
            by_hash[r["hash"]].append(r)
        keep = []
        for h, group in by_hash.items():
            labels = {r["label"] for r in group}
            if len(labels) > 1:                 # same picture, different labels → unreliable
                conflict_removed += len(group)
                continue
            # prefer a field image over a lab copy
            group.sort(key=lambda r: (r["field"] == "lab", r["dataset"]))
            keep.append(group[0])
            dup_removed += len(group) - 1
        rows = keep

    # ---- rare classes
    counts = Counter(r["label"] for r in rows)
    rare = {k for k, v in counts.items() if v < a.min_per_class}
    rows = [r for r in rows if r["label"] not in rare]

    # ---- split per (dataset, label)
    rng = random.Random(a.seed)
    groups = defaultdict(list)
    for r in rows:
        groups[(r["dataset"], r["label"])].append(r)
    train_only = {d["id"] for d in load_datasets() if d.get("train_only")}
    for (ds_id, _), g in groups.items():
        rng.shuffle(g)
        if ds_id in train_only:                 # augmented-only datasets: never in val/test
            for r in g:
                r["split"] = "train"
            continue
        n = len(g)
        n_test = max(1, round(n * a.test)) if n >= 10 else 0
        n_val = max(1, round(n * a.val)) if n >= 10 else 0
        for i, r in enumerate(g):
            r["split"] = "test" if i < n_test else "val" if i < n_test + n_val else "train"

    classes = sorted({r["label"] for r in rows})
    DATA.mkdir(exist_ok=True)
    with open(DATA / "manifest.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["path", "label", "dataset", "field", "weight", "split", "raw", "hash", "source"])
        w.writeheader()
        w.writerows(rows)
    dmap = {d["id"]: d for d in diseases}
    (DATA / "classes.json").write_text(json.dumps(
        [{"id": c, "crop": dmap[c]["crop"], "type": dmap[c]["type"]} for c in classes], indent=1))

    # ---- report
    REPORTS.mkdir(exist_ok=True)
    final = Counter(r["label"] for r in rows)
    split_c = Counter(r["split"] for r in rows)
    by_ds = Counter(r["dataset"] for r in rows)
    kb_ids = {d["id"] for d in diseases}
    lines = [
        "# Data preparation report", "",
        f"- images kept: **{len(rows):,}** (train {split_c['train']:,} · val {split_c['val']:,} · test {split_c['test']:,})",
        f"- classes: **{len(classes)}** of {len(kb_ids)} in the knowledge base",
        f"- unreadable or tiny images dropped: {bad:,}",
        f"- duplicates removed: {dup_removed:,}; images with conflicting labels removed: {conflict_removed:,}",
        f"- classes dropped for having < {a.min_per_class} images: {', '.join(sorted(rare)) or 'none'}", "",
        "## Images per dataset", "", "| dataset | images |", "|---|---|",
        *[f"| {k} | {v:,} |" for k, v in by_ds.most_common()], "",
        "## Images per class", "", "| class | images |", "|---|---|",
        *[f"| {k} | {v:,} |" for k, v in sorted(final.items())], "",
        "## Knowledge-base diseases with no training images (app can show advice, model cannot predict them)", "",
        ", ".join(sorted(kb_ids - set(classes))) or "none", "",
        "## Unmapped raw labels", "",
        "Add a synonym in kb/*.yaml, or a `map: {\"raw label\": crop.condition}` entry for the dataset in "
        "pipeline/datasets.yaml (map to an empty string to skip on purpose).", "",
        "| dataset | raw label | images |", "|---|---|---|",
        *[f"| {ds} | {lab} | {n:,} |" for ds, c in unmapped.items() for lab, n in c.most_common()], "",
        "## How raw labels were mapped (check these when adding a dataset)", "",
        "| dataset | raw label | → class | images |", "|---|---|---|---|",
        *[f"| {ds} | {raw} | {lab} | {n:,} |" for (ds, raw, lab), n in
          sorted(Counter((r["dataset"], r["raw"], r["label"]) for r in rows).items())][:1500],
    ]
    (REPORTS / "prepare_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"[done] {len(rows):,} images, {len(classes)} classes → {DATA / 'manifest.csv'}; see reports/prepare_report.md")
    if unmapped:
        print(f"       {sum(sum(c.values()) for c in unmapped.values()):,} images had unmapped labels (listed in the report)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile pipeline/train.py
"""Train the plant-disease classifier on data/manifest.csv.

Recommended two-stage recipe for the best phone model:
  1. Teacher (big, accurate):
       python pipeline/train.py --model convnextv2_tiny.fcmae_ft_in22k_in1k --img 288 --epochs 30 --out runs/teacher
  2. Student (small, fast, distilled from the teacher; this one goes into the app):
       python pipeline/train.py --model efficientnet_b0.ra_in1k --img 224 --epochs 40 \
           --teacher runs/teacher/best.pt --out runs/student

Quick CPU smoke test:
       python pipeline/train.py --model mobilenetv3_small_100 --epochs 1 --max-steps 20 --bs 16 --workers 2
"""
from __future__ import annotations

import argparse
import csv
import json
import math
import os
import time
from collections import Counter
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.transforms import v2 as T

from common import DATA_ROOT, ROOT

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


# ---------------------------------------------------------------- data
def read_manifest(path: Path):
    """Rows of manifest.csv; relative image paths are resolved against the manifest's folder."""
    base = Path(path).resolve().parent
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        if not os.path.isabs(r["path"]):
            r["path"] = str(base / r["path"])
    return rows


class ImgDS(Dataset):
    def __init__(self, rows, classes, tf):
        self.rows, self.tf = rows, tf
        self.idx = {c: i for i, c in enumerate(classes)}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        with Image.open(r["path"]) as im:
            im = im.convert("RGB")
        return self.tf(im), self.idx[r["label"]]


def train_tf(img):
    # Heavy, phone-realistic augmentation: lab photos must look like field photos.
    return T.Compose([
        T.RandomResizedCrop(img, scale=(0.30, 1.0), ratio=(0.75, 1.33), antialias=True),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(0.2),
        T.RandomApply([T.RandomRotation(25)], p=0.3),
        T.TrivialAugmentWide(),
        T.RandomApply([T.ColorJitter(0.35, 0.35, 0.3, 0.04)], p=0.6),
        T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.25),
        T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD),
        T.RandomErasing(p=0.25, value="random"),
    ])


def eval_tf(img):
    return T.Compose([T.Resize(int(img * 1.14), antialias=True), T.CenterCrop(img),
                      T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])


def sample_weights(rows, field_boost: float, balance_power: float):
    """Balance classes (count^-power), respect dataset weights and favour field photos."""
    cnt = Counter(r["label"] for r in rows)
    w = []
    for r in rows:
        x = float(r.get("weight") or 1.0) * cnt[r["label"]] ** -balance_power
        if r["field"] in ("field", "mixed"):
            x *= field_boost
        w.append(x)
    return torch.tensor(w, dtype=torch.double)


# ---------------------------------------------------------------- metrics
def macro_f1(y, p, n):
    cm = np.zeros((n, n), dtype=np.int64)
    np.add.at(cm, (y, p), 1)
    tp = np.diag(cm).astype(float)
    prec = tp / np.maximum(cm.sum(0), 1)
    rec = tp / np.maximum(cm.sum(1), 1)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-9)
    present = cm.sum(1) > 0
    return float(f1[present].mean()) if present.any() else 0.0


@torch.no_grad()
def predict(model, loader, dev, amp):
    model.eval()
    if len(loader.dataset) == 0:
        return torch.empty(0, 1), torch.empty(0, dtype=torch.long)
    logits, ys = [], []
    for x, y in loader:
        with torch.autocast(dev.type, enabled=amp):
            logits.append(model(x.to(dev, non_blocking=True)).float().cpu())
        ys.append(y)
    return torch.cat(logits), torch.cat(ys)


def fit_temperature(logits, y):
    """Temperature scaling on validation logits → calibrated confidences in the app."""
    t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([t], lr=0.05, max_iter=200)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / t.exp(), y)
        loss.backward()
        return loss
    opt.step(closure)
    return float(t.exp())


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--manifest", default=str(DATA_ROOT / "manifest.csv"))
    ap.add_argument("--model", default="efficientnet_b0.ra_in1k")
    ap.add_argument("--img", type=int, default=224)
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--warmup", type=float, default=1.0, help="warm-up epochs")
    ap.add_argument("--smoothing", type=float, default=0.1)
    ap.add_argument("--mixup", type=float, default=0.5, help="probability of mixup/cutmix per batch")
    ap.add_argument("--field-boost", type=float, default=2.0)
    ap.add_argument("--balance-power", type=float, default=0.5)
    ap.add_argument("--epoch-size", type=int, default=0, help="samples per epoch (0 = size of train set)")
    ap.add_argument("--teacher", default="", help="checkpoint of a teacher model for distillation")
    ap.add_argument("--kd-alpha", type=float, default=0.7)
    ap.add_argument("--kd-temp", type=float, default=2.0)
    ap.add_argument("--ema", type=float, default=0.9995)
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--max-steps", type=int, default=0, help="stop early (smoke tests)")
    ap.add_argument("--out", default=str(ROOT / "runs" / "model"))
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--time-limit", type=float, default=0, help="stop after this many minutes (keeps the best model)")
    ap.add_argument("--pretrained-file", default="", help="local ImageNet weights (.pth/.safetensors) for nodes without internet")
    a = ap.parse_args()

    torch.manual_seed(a.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = dev.type == "cuda"
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)

    rows = read_manifest(Path(a.manifest))
    classes = sorted({r["label"] for r in rows})
    tr = [r for r in rows if r["split"] == "train"]
    va = [r for r in rows if r["split"] == "val"]
    print(f"device={dev} classes={len(classes)} train={len(tr):,} val={len(va):,}")

    sampler = WeightedRandomSampler(sample_weights(tr, a.field_boost, a.balance_power),
                                    num_samples=a.epoch_size or len(tr), replacement=True)
    dl_tr = DataLoader(ImgDS(tr, classes, train_tf(a.img)), batch_size=a.bs, sampler=sampler,
                       num_workers=a.workers, pin_memory=amp, drop_last=True, persistent_workers=a.workers > 0)
    dl_va = DataLoader(ImgDS(va, classes, eval_tf(a.img)), batch_size=a.bs * 2,
                       num_workers=a.workers, pin_memory=amp)

    kw = {"pretrained_cfg_overlay": {"file": a.pretrained_file}} if a.pretrained_file else {}
    model = timm.create_model(a.model, pretrained=True, num_classes=len(classes), drop_rate=0.2,
                              drop_path_rate=0.1, **kw).to(dev)
    ema = None
    if a.ema > 0:
        try:
            ema = timm.utils.ModelEmaV3(model, decay=a.ema, use_warmup=True)
        except (AttributeError, TypeError):           # older timm
            ema = timm.utils.ModelEmaV2(model, decay=a.ema)

    teacher = None
    if a.teacher:
        ck = torch.load(a.teacher, map_location="cpu", weights_only=False)
        if ck["classes"] != classes:
            raise SystemExit("teacher was trained on different classes; retrain it on this manifest")
        teacher = timm.create_model(ck["model"], pretrained=False, num_classes=len(classes))
        teacher.load_state_dict(ck["state_dict"])
        teacher.eval().to(dev)
        t_img = ck["img"]
        print(f"distilling from {ck['model']} (img {t_img})")

    mix = timm.data.Mixup(mixup_alpha=0.2, cutmix_alpha=1.0, prob=a.mixup, label_smoothing=a.smoothing,
                          num_classes=len(classes)) if a.mixup > 0 else None
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
    steps_per_epoch = len(dl_tr)
    plan = {"total": a.epochs * steps_per_epoch}       # shortened below if a time limit applies
    warm = min(int(a.warmup * steps_per_epoch), 1000)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / max(warm, 1) if s < warm
        else max(0.0, 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(plan["total"] - warm, 1))))))
    scaler = torch.amp.GradScaler(enabled=amp)

    best, step, log = -1.0, 0, []
    t_start = time.time()
    out_of_time = lambda: a.time_limit and (time.time() - t_start) / 60 > a.time_limit
    va_field = np.array([r["field"] in ("field", "mixed") for r in va])
    for ep in range(a.epochs):
        model.train()
        t0, run = time.time(), 0.0
        for i, (x, y) in enumerate(dl_tr):
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            if mix:
                x, ys = mix(x, y)
            else:
                ys = F.one_hot(y, len(classes)).float() * (1 - a.smoothing) + a.smoothing / len(classes)
            with torch.autocast(dev.type, enabled=amp):
                logits = model(x)
                loss = torch.sum(-ys * F.log_softmax(logits.float(), -1), -1).mean()
                if teacher is not None:
                    with torch.no_grad():
                        xt = x if t_img == a.img else F.interpolate(x, size=t_img, mode="bilinear", align_corners=False)
                        tl = teacher(xt).float()
                    kd = F.kl_div(F.log_softmax(logits.float() / a.kd_temp, -1),
                                  F.softmax(tl / a.kd_temp, -1), reduction="batchmean") * a.kd_temp ** 2
                    loss = (1 - a.kd_alpha) * loss + a.kd_alpha * kd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            if ema:
                ema.update(model, step=step)
            run += loss.item()
            step += 1
            if a.time_limit and step == 100:
                # fit the cosine schedule into the time we have (keep ~12% for validation passes)
                per_step = (time.time() - t_start) / step
                fits = int(a.time_limit * 60 * 0.88 / per_step)
                if fits < plan["total"]:
                    plan["total"] = max(fits, warm + 1)
                    print(f"  {per_step:.3f}s/step → schedule set to {plan['total']:,} steps "
                          f"(≈{plan['total'] / steps_per_epoch:.1f} epochs) to fit {a.time_limit:.0f} min", flush=True)
            if i % 50 == 0:
                print(f"  ep {ep+1} step {i}/{steps_per_epoch} loss {run / (i + 1):.4f}", flush=True)
            if (a.max_steps and step >= a.max_steps) or out_of_time() or step >= plan["total"]:
                break

        net = ema.module if ema else model
        lg, yv = predict(net, dl_va, dev, amp)
        pv = lg.argmax(1).numpy() if len(yv) else np.zeros(0, dtype=np.int64)
        yv_np = yv.numpy()
        acc = float((pv == yv_np).mean()) if len(yv_np) else 0.0
        f1 = macro_f1(yv_np, pv, len(classes))
        f1_field = macro_f1(yv_np[va_field], pv[va_field], len(classes)) if va_field.any() else f1
        score = f1_field
        log.append({"epoch": ep + 1, "loss": run / max(i + 1, 1), "val_acc": acc, "val_f1": f1,
                    "val_f1_field": f1_field, "sec": round(time.time() - t0)})
        print(f"epoch {ep+1}/{a.epochs}  acc {acc:.4f}  macroF1 {f1:.4f}  field macroF1 {f1_field:.4f}  "
              f"({time.time() - t0:.0f}s)", flush=True)
        if score > best:
            best = score
            temp = fit_temperature(lg, yv) if len(yv) else 1.0
            torch.save({"model": a.model, "img": a.img, "mean": MEAN, "std": STD, "classes": classes,
                        "temperature": temp, "state_dict": net.state_dict(), "epoch": ep + 1,
                        "val": log[-1]}, out / "best.pt")
            print(f"  saved best (field macroF1 {best:.4f}, temperature {temp:.2f})")
        (out / "log.json").write_text(json.dumps(log, indent=1))
        if (a.max_steps and step >= a.max_steps) or out_of_time() or step >= plan["total"]:
            if out_of_time():
                print(f"time limit of {a.time_limit:.0f} min reached; stopping with the best model so far")
            break
    print(f"done. best field macro-F1 {best:.4f} → {out / 'best.pt'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile pipeline/datasets.yaml
# Public plant-disease image datasets used for training.
#
# host types and what `ref` means:
#   github     owner/repo (git clone, optional `subdir`)
#   zenodo     record id
#   mendeley   dataset id + `version`
#   dataverse  DOI (Harvard Dataverse)
#   kaggle     owner/dataset   (needs ~/.kaggle/kaggle.json)
#   kaggle_comp competition name (you must accept the rules on kaggle.com first)
#   hf         Hugging Face dataset repo id
#   manual     download yourself into data/raw/<id>/ (see `url`)
#
# field:   lab (plain background) · field (real conditions) · mixed
# licence: copy of what the source states; "check" means confirm on the source page.
# commercial: false for NC licences. Set `--commercial-only` in prepare.py to drop those.
#
# label: how classes are read
#   folder → class = name of the folder the image is in (optionally `label_depth`)
#   csv    → `csv: {file, image_col, label_col, image_dir}`
# crop: set for single-crop datasets so generic folder names ("healthy", "rust") map correctly.

datasets:
  # ---------------------------- broad multi-crop ----------------------------
  - id: plantvillage
    name: PlantVillage (color)
    host: github
    ref: spMohanty/PlantVillage-Dataset
    subdir: raw/color
    crops: [apple, blueberry, cherry, maize, grape, citrus, peach, pepper, potato, raspberry, soybean, squash, strawberry, tomato]
    images: 54305
    field: lab
    region: USA
    licence: CC BY-SA 4.0 (check)
    commercial: true
    label: folder
    weight: 0.5            # down-weighted: lab images dominate otherwise

  - id: plantdoc
    other_plants: true       # images of crops we do not cover → class other.unknown_plant
    name: PlantDoc (classification)
    host: github
    ref: pratikkayal/PlantDoc-Dataset
    crops: [apple, pepper, blueberry, cherry, maize, grape, peach, potato, raspberry, soybean, squash, strawberry, tomato]
    images: 2598
    field: field
    region: web images
    licence: CC BY 4.0 (check)
    commercial: true
    label: folder

  - id: plantwild
    other_plants: true       # images of crops we do not cover → class other.unknown_plant
    name: PlantWild (in-the-wild, 89+ classes)
    host: hf
    ref: uqtwei2/PlantWild
    crops: [many]
    field: field
    region: web images
    licence: CC BY-NC-ND 4.0
    commercial: false
    label: folder

  - id: plantseg
    other_plants: true       # images of crops we do not cover → class other.unknown_plant
    name: PlantSeg (in-the-wild, 115 classes)
    host: zenodo
    ref: "17719108"
    crops: [many]
    images: 7774
    field: field
    region: web images
    licence: CC BY-NC 4.0
    commercial: false
    label: folder
    note: Segmentation dataset; prepare.py uses the class folders of the images.

  - id: bd_crop_vegetable
    other_plants: true       # images of crops we do not cover → class other.unknown_plant
    name: Bangladesh crop & vegetable plant disease
    host: hf
    ref: Saon110/bd-crop-vegetable-plant-disease-dataset
    crops: [many]
    field: mixed
    region: Bangladesh
    licence: check
    commercial: unknown
    label: folder

  # ---------------------------- Africa ----------------------------
  - id: cassava_kaggle
    name: Cassava Leaf Disease (Makerere / NaCRRI, Uganda)
    host: kaggle_comp
    ref: cassava-leaf-disease-classification
    crop: cassava
    images: 21397
    field: field
    region: Uganda
    licence: competition rules (non-commercial research use)
    commercial: false
    label: csv
    csv: {file: train.csv, image_col: image_id, label_col: label, image_dir: train_images}

  - id: makerere_cassava
    name: Makerere University Cassava Image Dataset
    host: dataverse
    ref: doi:10.7910/DVN/T4RB0B
    crop: cassava
    field: field
    region: Uganda
    licence: CC BY 4.0 (check)
    commercial: true
    label: folder

  - id: makerere_maize
    name: Makerere University Maize Image Dataset (MSV, MLB)
    host: dataverse
    ref: doi:10.7910/DVN/LPGHKK
    crop: maize
    field: field
    region: Uganda
    licence: CC BY 4.0 (check)
    commercial: true
    label: folder

  - id: makerere_beans
    name: Makerere University Beans Image Dataset
    host: dataverse
    ref: doi:10.7910/DVN/TCKVEW
    crop: bean
    field: field
    region: Uganda
    licence: CC BY 4.0 (check)
    commercial: true
    label: folder

  - id: ibean
    name: iBean (AI Lab Makerere)
    host: hf
    ref: AI-Lab-Makerere/beans
    crop: bean
    images: 1295
    field: field
    region: Uganda
    licence: MIT
    commercial: true
    label: folder

  - id: tanzania_maize
    name: Labeled dataset of healthy and diseased maize (MLN, FAW) — Tanzania
    host: zenodo
    ref: "15685236"
    crop: maize
    field: field
    region: Tanzania
    licence: check
    commercial: unknown
    label: folder

  - id: fieldplant
    name: FieldPlant (maize, cassava, tomato) — Cameroon
    host: manual
    url: https://universe.roboflow.com  (search "FieldPlant")
    crops: [maize, cassava, tomato]
    images: 5170
    field: field
    region: Cameroon
    licence: check
    commercial: unknown
    label: folder
    note: Export from Roboflow as "folder" classification format into data/raw/fieldplant/.

  - id: banana_tanzania
    name: Banana leaves imagery (Black Sigatoka, Fusarium wilt) — Tanzania
    host: manual
    url: https://dspace.nm-aist.ac.tz/handle/20.500.12479/3003
    crop: banana
    field: field
    region: Tanzania
    licence: check
    commercial: unknown
    label: folder

  - id: cgiar_wheat_rust
    name: CGIAR Computer Vision for Crop Disease (wheat rust) — Ethiopia
    host: manual
    url: https://zindi.africa/competitions/iclr-workshop-challenge-1-cgiar-computer-vision-for-crop-disease/data
    crop: wheat
    field: field
    region: Ethiopia
    licence: Zindi competition rules
    commercial: false
    label: folder
    note: Log in to Zindi, accept the rules, download, unzip into data/raw/cgiar_wheat_rust/.

  - id: wheat_diseases_zenodo
    name: Wheat Plant Diseases Dataset
    host: zenodo
    ref: "15056570"
    crop: wheat
    field: mixed
    licence: check
    commercial: unknown
    label: folder

  - id: wheat_leaf_mendeley
    name: Wheat Leaf Dataset
    host: mendeley
    ref: wgd66f8n6h
    version: 1
    crop: wheat
    field: field
    licence: CC BY 4.0 (check)
    commercial: true
    label: folder

  # ---------------------------- single crops elsewhere ----------------------------
  - id: paddy_doctor
    name: Paddy Doctor (10 rice classes)
    host: kaggle_comp
    ref: paddy-disease-classification
    crop: rice
    images: 10407
    field: field
    region: India
    licence: competition rules
    commercial: false
    label: folder
    label_dir: train_images

  - id: plant_pathology_2021
    name: Plant Pathology 2021 (apple, FGVC8)
    host: kaggle_comp
    ref: plant-pathology-2021-fgvc8
    crop: apple
    images: 18632
    field: field
    region: USA
    licence: competition rules
    commercial: false
    label: csv
    csv: {file: train.csv, image_col: image, label_col: labels, image_dir: train_images, multi_label_sep: " "}

  - id: corn_kaggle
    name: Corn or Maize Leaf Disease
    host: kaggle
    ref: smaranjitghose/corn-or-maize-leaf-disease-dataset
    crop: maize
    images: 4188
    field: mixed
    licence: check
    commercial: unknown
    label: folder

  - id: potato_uncontrolled
    name: Potato Leaf Disease Dataset in Uncontrolled Environment
    host: mendeley
    ref: ptz377bwb8
    version: 1
    crop: potato
    images: 3076
    field: field
    region: Indonesia
    licence: CC BY 4.0 (check)
    commercial: true
    label: folder

  - id: rocole
    name: RoCoLe robusta coffee leaves
    host: mendeley
    ref: c5yvn32dzg
    version: 2
    crop: coffee
    images: 1560
    field: field
    region: Ecuador
    licence: CC BY 4.0 (check)
    commercial: true
    label: folder

  - id: bracol
    name: BRACOL Arabica coffee leaves
    host: mendeley
    ref: yy2k5y8mxg
    version: 1
    crop: coffee
    field: field
    region: Brazil
    licence: CC BY 4.0 (check)
    commercial: true
    label: folder

  - id: mangoleafbd
    name: MangoLeafBD
    host: mendeley
    ref: hxsnvwty3r
    version: 1
    crop: mango
    images: 4000
    field: field
    region: Bangladesh
    licence: CC BY 4.0 (check)
    commercial: true
    label: folder

  - id: groundnut_leaf
    name: Groundnut leaf dataset
    host: mendeley
    ref: x6x5jkk873
    version: 2
    crop: groundnut
    field: field
    licence: CC BY 4.0 (check)
    commercial: true
    label: folder
    map: {"LEAF SPOT (EARLY AND LATE)": ""}   # early and late leaf spot mixed in one folder: cannot be split

  - id: sorghum_disease
    name: Sorghum Disease Image Dataset
    host: mendeley
    ref: fgb3cmfrg2
    version: 1
    crop: sorghum
    field: field
    licence: CC BY 4.0 (check)
    commercial: true
    label: folder

  # ---------------------- data round 3 (Oct 2026): fruits ----------------------
  # `exclude`: skip images whose path contains any of these words (case-insensitive). Used to drop the
  # augmented copies some datasets ship: flipped/rotated copies of the same photo would land in both
  # train and test and make the scores look better than they are.
  # `label_join`: class = all folder names joined ("Mango/Healthy Fruit" → "Mango Healthy Fruit"),
  # for multi-crop datasets whose class folders do not repeat the crop name.
  - id: mango_dhds
    name: "MangoDHDS: diseased and healthy mango fruits"
    host: mendeley
    ref: b4nrw5hyyc
    version: 1
    crop: mango
    images: 1500
    field: mixed
    region: India
    licence: CC BY 4.0
    commercial: true
    label: folder

  - id: mango_amrapali
    name: Amrapali mango fruit diseases
    host: mendeley
    ref: ypttkp5fb5
    version: 1
    crop: mango
    images: 3500
    field: field
    region: India
    licence: CC BY 4.0 (check)
    commercial: true
    label: folder
    exclude: [augment]

  - id: mango_leaf_field
    name: Mango leaf dataset (natural conditions)
    host: mendeley
    ref: fn8dgf4hb5
    version: 1
    crop: mango
    images: 1319
    field: field
    region: Bangladesh
    licence: CC BY 4.0
    commercial: true
    label: folder
    exclude: [augment]
    map: {"Die Black": mango.die_back}

  - id: abcgmp
    name: ABCGMP fruit and leaf diseases (apple, banana, citrus, guava, mango, papaya)
    host: mendeley
    ref: fhbvmpcyy2
    version: 3
    field: field
    region: India (Rajasthan, Jammu)
    licence: CC BY 4.0
    commercial: true
    label: folder
    label_join: true
    exclude: [augment, disease fruit, disease_fruit, diseased fruit]   # "disease fruit" does not say which disease

  - id: guava_fruit_leaf
    name: Guava fruits and leaves disease dataset
    host: mendeley
    ref: fspx44mwfp
    version: 1
    crop: guava
    images: 3432
    field: mixed
    region: Bangladesh
    licence: CC BY 4.0
    commercial: true
    label: folder
    exclude: [augment]
    map: {"Dot": ""}        # unclear cause, not used

  - id: guava_fruit_bd
    name: Guava fruit disease dataset (orchards)
    host: mendeley
    ref: bkdkc4n835
    version: 1
    crop: guava
    images: 473
    field: field
    region: Bangladesh
    licence: CC BY 4.0
    commercial: true
    label: folder
    train_only: true        # ships only augmented copies: train on them, never test on them

  - id: papaya_leaf
    name: Papaya leaf disease image dataset
    host: mendeley
    ref: 3kwgxg4stb
    version: 1
    crop: papaya
    images: 3626
    field: field
    region: Bangladesh
    licence: CC BY 4.0
    commercial: true
    label: folder
    exclude: [augment]

In [ ]:
%%writefile kb/cereals.yaml
diseases:
# ============================== MAIZE ==============================
- id: maize.healthy
  crop: maize
  type: healthy
  name: {en: Healthy maize, am: ጤናማ በቆሎ}
  synonyms: [corn healthy, maize healthy, "Corn_(maize)___healthy", healthy maize, corn___healthy]

- id: maize.northern_leaf_blight
  crop: maize
  type: fungal
  severity: medium
  name: {en: Northern leaf blight (turcicum blight)}
  scientific: Exserohilum turcicum
  synonyms: [northern leaf blight, northern corn leaf blight, "Corn_(maize)___Northern_Leaf_Blight", maize leaf blight, MLB, turcicum leaf blight, leaf blight, blight, nlb]
  symptoms:
    - Long (3–15 cm), cigar-shaped grey-green to tan lesions on leaves
    - Lesions start on lower leaves and move up; dark spores can appear in the centre in humid weather
    - Severe infection before tasselling dries leaves early and reduces yield
  spread: Survives on maize residue; spores spread by wind and rain splash. Favoured by moderate temperatures (18–27 °C), heavy dew and long wet periods.
  cultural:
    - Plant resistant or tolerant varieties recommended for your area
    - Rotate with a non-grass crop (beans, soybean, groundnut) for at least one season
    - Bury or remove infected residue after harvest
    - Avoid very dense planting so leaves dry faster
  biological:
    - Trichoderma-based products can reduce inoculum on residue (moderate effect)
  chemical:
    - Only if lesions reach the ear leaf before or at tasselling on a susceptible variety
    - Triazoles (e.g. propiconazole, tebuconazole) or strobilurin + triazole mixtures
  prevention: Resistant varieties and rotation give the best long-term control.

- id: maize.gray_leaf_spot
  crop: maize
  type: fungal
  severity: medium
  name: {en: Gray leaf spot}
  scientific: Cercospora zeae-maydis / C. zeina
  synonyms: [gray leaf spot, grey leaf spot, "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", cercospora leaf spot, gls]
  symptoms:
    - Narrow, rectangular grey to tan lesions limited by leaf veins
    - Lesions join and kill large leaf areas during grain fill
  spread: Survives on residue; favoured by warm, very humid weather, mist and reduced tillage. Common in East and Southern African highlands.
  cultural:
    - Use tolerant hybrids
    - Rotate away from maize and manage residue
    - Avoid continuous maize under minimum tillage in high-risk zones
  biological:
    - No reliable biocontrol yet; focus on resistance and rotation
  chemical:
    - Strobilurin or strobilurin + triazole fungicides at early lesion development on susceptible varieties
  prevention: Choose tolerant varieties for humid, high-altitude areas.

- id: maize.common_rust
  crop: maize
  type: fungal
  severity: low
  name: {en: Common rust}
  scientific: Puccinia sorghi
  synonyms: [common rust, "Corn_(maize)___Common_rust_", maize rust, corn rust, rust]
  symptoms:
    - Small, oval, cinnamon-brown powdery pustules on both leaf surfaces
    - Pustules turn dark brown-black late in the season
  spread: Wind-blown spores; favoured by cool (16–23 °C), humid conditions.
  cultural:
    - Plant resistant hybrids; most commercial hybrids have good tolerance
    - Plant early to avoid peak spore periods
  biological:
    - None proven at field scale
  chemical:
    - Usually not needed. On susceptible seed crops, triazole or strobilurin fungicides at early infection
  prevention: Resistant varieties.

- id: maize.streak_virus
  crop: maize
  type: viral
  severity: high
  name: {en: Maize streak virus (MSV)}
  scientific: Maize streak virus (Mastrevirus), spread by Cicadulina leafhoppers
  synonyms: [maize streak virus, msv, streak virus, maize streak]
  symptoms:
    - Narrow, broken yellow-white streaks running along the leaf veins
    - Plants infected young are stunted and may produce no cobs
  spread: Transmitted only by Cicadulina leafhoppers from grasses and earlier maize. Worst in late-planted fields and dry spells. Native to Africa.
  cultural:
    - Plant MSV-resistant varieties (widely available in Africa)
    - Plant early and at the same time as neighbours
    - Remove infected seedlings early and control grass weeds around the field
  biological:
    - Natural enemies of leafhoppers; avoid broad-spectrum insecticides that kill them
  chemical:
    - No chemical cure for the virus
    - Systemic insecticide seed treatment (e.g. imidacloprid, thiamethoxam) can reduce early leafhopper feeding in high-risk areas
  prevention: Resistant varieties plus synchronized early planting.

- id: maize.lethal_necrosis
  crop: maize
  type: viral
  severity: critical
  name: {en: Maize lethal necrosis (MLN)}
  scientific: Maize chlorotic mottle virus + a potyvirus (usually SCMV)
  synonyms: [maize lethal necrosis, mln, lethal necrosis]
  symptoms:
    - Yellow mottling starting at leaf bases, then drying from the leaf margins inward ("firing")
    - Dead heart, premature drying, small or empty cobs; whole fields can die
  spread: Thrips, rootworms and aphids spread the viruses; MCMV is also seed-borne. Spread rapidly in East Africa since 2011.
  cultural:
    - Use certified MLN-free seed and MLN-tolerant varieties
    - Do not plant maize continuously; observe maize-free periods where advised
    - Uproot and destroy infected plants early; control weeds
    - Rotate with legumes, potato or other non-cereals
  biological:
    - No biocontrol for the virus; conserve vector enemies
  chemical:
    - No chemical cure
    - Vector control (seed treatment, insecticides against thrips/aphids) only as advised by extension
  prevention: Certified seed, rotation and community-wide maize-free periods.
  notes: Notifiable in several countries. Report suspected MLN to your extension or plant-protection office.

- id: maize.fall_armyworm
  crop: maize
  type: pest
  severity: high
  name: {en: Fall armyworm damage}
  scientific: Spodoptera frugiperda
  synonyms: [fall armyworm, faw, armyworm, fall_armyworm]
  symptoms:
    - Window-pane feeding (scraped, see-through patches) on young leaves
    - Ragged holes, sawdust-like droppings (frass) in the leaf funnel (whorl)
    - Caterpillars with an inverted "Y" on the head and four dots in a square near the tail
  spread: Moths fly long distances; spread across Africa since 2016. Several generations per season.
  cultural:
    - Scout twice a week in the first 6 weeks; act early
    - Hand-pick egg masses and young larvae; put sand, ash or soil mixed with lime in the whorl
    - Plant early and at the same time; intercrop with legumes; "push-pull" with desmodium and Brachiaria
  biological:
    - Products based on Bt (Bacillus thuringiensis), Metarhizium or SfMNPV virus
    - Protect natural enemies (parasitic wasps, earwigs, ants)
  chemical:
    - Only if damage exceeds local thresholds (about 20% of young plants damaged)
    - Use recommended actives such as emamectin benzoate, spinetoram or chlorantraniliprole; spray into the whorl; rotate modes of action
  prevention: Regular scouting, push-pull, early synchronized planting.

# ============================== WHEAT ==============================
- id: wheat.healthy
  crop: wheat
  type: healthy
  name: {en: Healthy wheat, am: ጤናማ ስንዴ}
  synonyms: [healthy wheat, wheat healthy, healthy]

- id: wheat.stem_rust
  crop: wheat
  type: fungal
  severity: critical
  name: {en: Stem rust (black rust)}
  scientific: Puccinia graminis f. sp. tritici
  synonyms: [stem rust, black rust, wheat stem rust, stem_rust]
  symptoms:
    - Elongated, dark reddish-brown pustules on stems, leaf sheaths and leaves that tear the plant skin
    - Pustules turn black late in the season; stems weaken and lodge
  spread: Wind-blown spores over long distances; warm days (20–30 °C) with wet nights. Ug99 races arose in East Africa.
  cultural:
    - Grow resistant varieties released for your country (check for Ug99 resistance)
    - Remove volunteer wheat and avoid very late planting
  biological:
    - None proven at field scale
  chemical:
    - Triazoles (e.g. tebuconazole, propiconazole) or triazole + strobilurin at first sign on susceptible varieties
  prevention: Resistant varieties; surveillance with extension services.
  notes: Report new outbreaks to extension; Ug99-lineage races threaten many varieties.

- id: wheat.leaf_rust
  crop: wheat
  type: fungal
  severity: medium
  name: {en: Leaf rust (brown rust)}
  scientific: Puccinia triticina
  synonyms: [leaf rust, brown rust, wheat leaf rust, leaf_rust]
  symptoms:
    - Small, round, orange-brown pustules scattered on the upper leaf surface
  spread: Wind-blown spores; mild temperatures (15–22 °C) and dew.
  cultural:
    - Resistant varieties; remove volunteer wheat; avoid excess nitrogen
  biological:
    - None proven at field scale
  chemical:
    - Triazole fungicides when pustules appear on upper leaves before flowering
  prevention: Resistant varieties.

- id: wheat.yellow_rust
  crop: wheat
  type: fungal
  severity: high
  name: {en: Yellow rust (stripe rust)}
  scientific: Puccinia striiformis f. sp. tritici
  synonyms: [yellow rust, stripe rust, wheat yellow rust, yellow_rust]
  symptoms:
    - Yellow-orange pustules in long stripes between the leaf veins
    - Leaves turn yellow and dry; heads can be infected
  spread: Cool (10–15 °C), moist weather; a major problem in the Ethiopian and Kenyan highlands.
  cultural:
    - Resistant varieties; avoid early sowing in hot-spot areas; remove volunteers
  biological:
    - None proven at field scale
  chemical:
    - Triazoles (e.g. tebuconazole, propiconazole) at first stripes on susceptible varieties
  prevention: Resistant varieties and early scouting in cool, wet seasons.

- id: wheat.septoria
  crop: wheat
  type: fungal
  severity: medium
  name: {en: Septoria leaf blotch}
  scientific: Zymoseptoria tritici
  synonyms: [septoria, septoria tritici blotch, septoria leaf blotch, wheat septoria]
  symptoms:
    - Brown, irregular blotches with tiny black dots (fruiting bodies) inside
    - Starts on lower leaves and moves up in rain
  spread: Rain splash from residue; cool, wet seasons.
  cultural:
    - Rotation, residue management, tolerant varieties, avoid very early sowing
  biological:
    - None proven at field scale
  chemical:
    - Triazole or SDHI-containing mixtures at flag-leaf stage on susceptible varieties
  prevention: Rotation and tolerant varieties.

- id: wheat.powdery_mildew
  crop: wheat
  type: fungal
  severity: low
  name: {en: Powdery mildew}
  scientific: Blumeria graminis f. sp. tritici
  synonyms: [wheat powdery mildew, mildew]
  symptoms:
    - White to grey fluffy patches on leaves and stems that rub off
  spread: Wind; dense crops, high humidity and high nitrogen.
  cultural:
    - Resistant varieties; balanced nitrogen; avoid very dense sowing
  biological:
    - Sulphur products are accepted in organic systems
  chemical:
    - Sulphur or triazole fungicides if more than a few upper leaves are covered before flowering
  prevention: Resistant varieties.

# ============================== RICE ==============================
- id: rice.healthy
  crop: rice
  type: healthy
  name: {en: Healthy rice, am: ጤናማ ሩዝ}
  synonyms: [normal, healthy rice, rice healthy, healthy]

- id: rice.blast
  crop: rice
  type: fungal
  severity: high
  name: {en: Rice blast}
  scientific: Magnaporthe oryzae
  synonyms: [blast, rice blast, leaf blast, neck blast, leafblast]
  symptoms:
    - Diamond-shaped lesions with grey centres and brown edges on leaves
    - 'Neck blast: the stem below the panicle turns black and the panicle breaks or stays empty'
  spread: Airborne spores; long leaf wetness, cool nights, high nitrogen. Major threat in upland rice in Africa.
  cultural:
    - Resistant varieties; split nitrogen applications; avoid excess nitrogen
    - Keep fields flooded where possible; destroy infected straw
  biological:
    - Silicon fertilisers and Pseudomonas / Bacillus seed treatments reduce severity
  chemical:
    - Tricyclazole, azoxystrobin or isoprothiolane at booting if blast is present
  prevention: Resistant varieties and balanced fertiliser.

- id: rice.brown_spot
  crop: rice
  type: fungal
  severity: medium
  name: {en: Brown spot}
  scientific: Bipolaris oryzae
  synonyms: [brown spot, brownspot, rice brown spot]
  symptoms:
    - Oval brown spots with a grey centre and yellow halo on leaves and grains
  spread: Seed-borne; worst on poor, nutrient-deficient soils and in drought stress.
  cultural:
    - Use clean seed; correct potassium, silicon and nitrogen deficiencies
  biological:
    - Hot-water or Trichoderma seed treatment
  chemical:
    - Seed treatment with a recommended fungicide; foliar triazole in severe cases
  prevention: Healthy soil fertility and clean seed.

- id: rice.bacterial_leaf_blight
  crop: rice
  type: bacterial
  severity: high
  name: {en: Bacterial leaf blight}
  scientific: Xanthomonas oryzae pv. oryzae
  synonyms: [bacterial leaf blight, bacterial_leaf_blight, blb, bacterial blight]
  symptoms:
    - Wavy yellow to white stripes starting at leaf tips or edges, drying to grey
    - Milky bacterial ooze drops in the morning
  spread: Water, wind-driven rain, wounds, infected stubble; warm and flooded conditions.
  cultural:
    - Resistant varieties; balanced nitrogen; drain fields periodically; clean tools; remove stubble
  biological:
    - Bacillus-based products have limited effect
  chemical:
    - Copper products have limited value; resistance and hygiene are the real control
  prevention: Resistant varieties.

- id: rice.bacterial_leaf_streak
  crop: rice
  type: bacterial
  severity: medium
  name: {en: Bacterial leaf streak}
  scientific: Xanthomonas oryzae pv. oryzicola
  synonyms: [bacterial leaf streak, bacterial_leaf_streak, bls]
  symptoms:
    - Narrow, water-soaked streaks between veins that turn yellow-brown, with amber droplets
  spread: Rain splash, wind, irrigation water and seed.
  cultural:
    - Clean seed, resistant varieties, avoid excess nitrogen, remove infected stubble
  biological:
    - None reliable
  chemical:
    - Copper products have limited value
  prevention: Clean seed and resistance.

- id: rice.bacterial_panicle_blight
  crop: rice
  type: bacterial
  severity: medium
  name: {en: Bacterial panicle blight}
  scientific: Burkholderia glumae
  synonyms: [bacterial panicle blight, bacterial_panicle_blight]
  symptoms:
    - Grains discoloured straw to grey with a brown base; panicles stay upright because grains are empty
  spread: Seed-borne; very hot nights during flowering.
  cultural:
    - Clean seed; avoid planting so that flowering falls in the hottest period; moderate nitrogen
  biological:
    - None reliable
  chemical:
    - No effective chemical control in most countries
  prevention: Clean seed and planting date.

- id: rice.tungro
  crop: rice
  type: viral
  severity: high
  name: {en: Rice tungro}
  scientific: Rice tungro bacilliform + spherical viruses (green leafhoppers)
  synonyms: [tungro, rice tungro]
  symptoms:
    - Yellow-orange leaves from the tip down, stunting, fewer tillers
  spread: Green leafhoppers.
  cultural:
    - Resistant varieties; synchronised planting; remove infected stubble and volunteers
  biological:
    - Protect leafhopper predators (spiders, mirid bugs)
  chemical:
    - No chemical cure; insecticides against leafhoppers only on extension advice
  prevention: Synchronised planting and resistance.

- id: rice.hispa
  crop: rice
  type: pest
  severity: medium
  name: {en: Rice hispa damage}
  scientific: Dicladispa armigera
  synonyms: [hispa, rice hispa]
  symptoms:
    - White parallel feeding streaks and blotch mines on leaves; small spiny dark beetles
  spread: Adult beetles fly between fields; favoured by excess nitrogen and heavy rain.
  cultural:
    - Clip and destroy leaf tips with eggs before transplanting; avoid excess nitrogen
  biological:
    - Conserve natural enemies (parasitoid wasps)
  chemical:
    - Only at high infestations, using recommended insecticides
  prevention: Balanced fertilisation and early detection.

- id: rice.dead_heart
  crop: rice
  type: pest
  severity: medium
  name: {en: Dead heart (stem borer)}
  scientific: Rice stem borers (e.g. Scirpophaga, Chilo, Maliarpha spp.)
  synonyms: [dead heart, dead_heart, deadheart, stem borer]
  symptoms:
    - Central shoot dries and pulls out easily at the vegetative stage; white empty heads later
  spread: Moths lay eggs on leaves; larvae bore into stems.
  cultural:
    - Harvest close to the ground and destroy stubble; synchronised planting; remove egg masses
  biological:
    - Trichogramma egg parasitoids; conserve natural enemies
  chemical:
    - Only if above local thresholds; use recommended insecticides
  prevention: Stubble destruction and synchronised planting.

- id: rice.downy_mildew
  crop: rice
  type: oomycete
  severity: low
  name: {en: Downy mildew}
  scientific: Sclerophthora macrospora
  synonyms: [downy mildew, downy_mildew]
  symptoms:
    - Twisted, thickened yellowish leaves; stunted plants; distorted panicles
  spread: Waterlogged or flooded seedlings in cool weather.
  cultural:
    - Good drainage at seedling stage; remove infected plants; control grassy weeds
  biological:
    - None reliable
  chemical:
    - Rarely justified
  prevention: Good water management at seedling stage.

# ============================== SORGHUM ==============================
- id: sorghum.healthy
  crop: sorghum
  type: healthy
  name: {en: Healthy sorghum, am: ጤናማ ማሽላ}
  synonyms: [healthy sorghum, sorghum healthy, healthy]

- id: sorghum.anthracnose
  crop: sorghum
  type: fungal
  severity: high
  name: {en: Sorghum anthracnose}
  scientific: Colletotrichum sublineola
  synonyms: [anthracnose, sorghum anthracnose]
  symptoms:
    - Small round to elliptical spots with tan or red centres and dark margins; black spore dots in the centre
    - Stalk rot and red discoloration inside stems in severe cases
  spread: Residue, seed, rain splash; warm, humid weather.
  cultural:
    - Resistant varieties, rotation, residue removal, clean seed
  biological:
    - None proven at field scale
  chemical:
    - Seed treatment; foliar triazole or strobilurin only on susceptible varieties in high pressure
  prevention: Resistant varieties and rotation.

- id: sorghum.leaf_blight
  crop: sorghum
  type: fungal
  severity: medium
  name: {en: Sorghum leaf blight}
  scientific: Exserohilum turcicum
  synonyms: [leaf blight, sorghum leaf blight, northern leaf blight]
  symptoms:
    - Long, elliptical reddish or tan lesions on leaves
  spread: Residue and wind-blown spores; humid, moderate temperatures.
  cultural:
    - Resistant varieties, rotation, residue management
  biological:
    - None proven at field scale
  chemical:
    - Rarely economic; triazoles on seed crops
  prevention: Resistant varieties.

- id: sorghum.rust
  crop: sorghum
  type: fungal
  severity: low
  name: {en: Sorghum rust}
  scientific: Puccinia purpurea
  synonyms: [sorghum rust, rust]
  symptoms:
    - Small reddish-brown to purple pustules on both leaf surfaces
  spread: Wind-blown spores.
  cultural:
    - Resistant varieties; early planting
  biological:
    - None proven
  chemical:
    - Rarely needed
  prevention: Resistant varieties.

In [ ]:
%%writefile kb/crops.yaml
# Crops the app knows. `group` drives the crop picker in the app.
# Missing translations fall back to English.
crops:
  # ---- cereals ----
  - {id: maize,      group: cereal,    name: {en: Maize (corn), am: በቆሎ, fr: Maïs, sw: Mahindi, it: Mais}}
  - {id: wheat,      group: cereal,    name: {en: Wheat, am: ስንዴ, fr: Blé, sw: Ngano, it: Frumento}}
  - {id: rice,       group: cereal,    name: {en: Rice, am: ሩዝ, fr: Riz, sw: Mpunga, it: Riso}}
  - {id: sorghum,    group: cereal,    name: {en: Sorghum, am: ማሽላ, fr: Sorgho, sw: Mtama, it: Sorgo}}
  # ---- legumes ----
  - {id: bean,       group: legume,    name: {en: Common bean, am: ቦሎቄ, fr: Haricot, sw: Maharage, it: Fagiolo}}
  - {id: soybean,    group: legume,    name: {en: Soybean, am: አኩሪ አተር, fr: Soja, sw: Soya, it: Soia}}
  - {id: groundnut,  group: legume,    name: {en: Groundnut (peanut), am: ለውዝ, fr: Arachide, sw: Karanga, it: Arachide}}
  # ---- roots & tubers ----
  - {id: cassava,    group: root,      name: {en: Cassava, am: ካሳቫ, fr: Manioc, sw: Muhogo, it: Manioca}}
  - {id: potato,     group: root,      name: {en: Potato, am: ድንች, fr: Pomme de terre, sw: Viazi, it: Patata}}
  # ---- vegetables ----
  - {id: tomato,     group: vegetable, name: {en: Tomato, am: ቲማቲም, fr: Tomate, sw: Nyanya, it: Pomodoro}}
  - {id: pepper,     group: vegetable, name: {en: Pepper (capsicum), am: ቃሪያ, fr: Poivron, sw: Pilipili hoho, it: Peperone}}
  - {id: squash,     group: vegetable, name: {en: Squash / pumpkin, am: ዱባ, fr: Courge, sw: Maboga, it: Zucca}}
  # ---- fruits ----
  - {id: banana,     group: fruit,     name: {en: Banana, am: ሙዝ, fr: Banane, sw: Ndizi, it: Banano}}
  - {id: mango,      group: fruit,     name: {en: Mango, am: ማንጎ, fr: Mangue, sw: Embe, it: Mango}}
  - {id: citrus,     group: fruit,     name: {en: Citrus (orange), am: ብርቱካን, fr: Agrumes, sw: Chungwa, it: Agrumi (arancio)}}
  - {id: apple,      group: fruit,     name: {en: Apple, am: ፖም, fr: Pomme, sw: Tufaha, it: Melo}}
  - {id: grape,      group: fruit,     name: {en: Grape, am: ወይን, fr: Raisin, sw: Zabibu, it: Vite}}
  - {id: peach,      group: fruit,     name: {en: Peach, am: ኮክ, fr: Pêche, sw: Pichi, it: Pesco}}
  - {id: cherry,     group: fruit,     name: {en: Cherry, am: ቼሪ, fr: Cerise, it: Ciliegio}}
  - {id: strawberry, group: fruit,     name: {en: Strawberry, am: እንጆሪ, fr: Fraise, sw: Stroberi, it: Fragola}}
  - {id: blueberry,  group: fruit,     name: {en: Blueberry, fr: Myrtille, it: Mirtillo}}
  - {id: raspberry,  group: fruit,     name: {en: Raspberry, fr: Framboise, it: Lampone}}
  - {id: guava,      group: fruit,     name: {en: Guava, am: ዘይቱን, fr: Goyave, sw: Pera, it: Guava}}
  - {id: papaya,     group: fruit,     name: {en: Papaya, am: ፓፓያ, fr: Papaye, sw: Papai, it: Papaya}}
  # ---- cash / beverage ----
  - {id: coffee,     group: cash,      name: {en: Coffee, am: ቡና, fr: Café, sw: Kahawa, it: Caffè}}
  # ---- anything else ----
  - {id: other,      group: other,     name: {en: Other plant, am: ሌላ ተክል, fr: Autre plante, sw: Mmea mwingine, it: Altra pianta}}

groups:
  cereal:    {en: Cereals, am: የእህል ሰብሎች, fr: Céréales, sw: Nafaka, it: Cereali}
  legume:    {en: Legumes, am: ጥራጥሬዎች, fr: Légumineuses, sw: Mikunde, it: Legumi}
  root:      {en: Roots & tubers, am: የሥራ ሥር ሰብሎች, fr: Racines et tubercules, sw: Mazao ya mizizi, it: Radici e tuberi}
  vegetable: {en: Vegetables, am: አትክልቶች, fr: Légumes, sw: Mboga, it: Ortaggi}
  fruit:     {en: Fruits, am: ፍራፍሬዎች, fr: Fruits, sw: Matunda, it: Frutta}
  cash:      {en: Cash crops, am: የገበያ ሰብሎች, fr: Cultures de rente, sw: Mazao ya biashara, it: Colture da reddito}
  other:     {en: Other, am: ሌላ, fr: Autre, sw: Nyingine, it: Altro}

In [ ]:
%%writefile kb/fruits_cash.yaml
diseases:
# ============================== BANANA ==============================
- id: banana.healthy
  crop: banana
  type: healthy
  name: {en: Healthy banana, am: ጤናማ ሙዝ}
  synonyms: [healthy, banana healthy, healthy banana]

- id: banana.black_sigatoka
  crop: banana
  type: fungal
  severity: high
  name: {en: Black Sigatoka (black leaf streak)}
  scientific: Pseudocercospora fijiensis
  synonyms: [black sigatoka, sigatoka, black leaf streak, bs]
  symptoms:
    - Small reddish-brown streaks parallel to leaf veins that become black spots with grey centres
    - Large dead leaf areas; bunches ripen early and are small
  spread: Wind and rain; warm, wet weather.
  cultural:
    - Cut and remove (de-leaf) infected leaves and leaf parts every 1–2 weeks
    - Good drainage, correct spacing, weed control; tolerant varieties
  biological:
    - Bacillus-based products in integrated programmes
  chemical:
    - Mineral oil plus protectants (mancozeb, chlorothalonil) or systemics (triazoles, strobilurins) in commercial plantations; rotate groups
  prevention: Regular leaf sanitation.

- id: banana.fusarium_wilt
  crop: banana
  type: fungal
  severity: critical
  name: {en: Fusarium wilt (Panama disease)}
  scientific: Fusarium oxysporum f. sp. cubense (incl. Tropical Race 4)
  synonyms: [fusarium wilt, panama disease, fusarium, fwb]
  symptoms:
    - Older leaves turn yellow from the edges and collapse around the stem like a skirt
    - Splitting of the pseudostem base; reddish-brown streaks inside when cut
  spread: Soil, infected suckers, water, tools, shoes and vehicles. Survives decades in soil.
  cultural:
    - Plant clean tissue-culture plantlets of resistant varieties
    - Do not move soil, suckers or tools from infected fields; disinfect boots and tools
    - Destroy infected plants on the spot; do not replant bananas there
  biological:
    - Soil health and Trichoderma may reduce disease; no cure
  chemical:
    - No effective chemical cure
  prevention: Quarantine, clean planting material, resistant varieties.
  notes: Tropical Race 4 is a quarantine pest. Report suspected cases to the plant-protection service.

- id: banana.xanthomonas_wilt
  crop: banana
  type: bacterial
  severity: critical
  name: {en: Banana Xanthomonas wilt (BXW)}
  scientific: Xanthomonas vasicola pv. musacearum
  synonyms: [banana xanthomonas wilt, bxw, xanthomonas wilt, banana bacterial wilt]
  symptoms:
    - Yellowing and wilting leaves; fruits ripen early and unevenly with brown stains inside; male bud shrivels
    - Yellow bacterial ooze from cut stems
  spread: Insects visiting the male flower, contaminated tools, infected suckers. Major threat in East and Central Africa.
  cultural:
    - Remove the male bud with a forked stick as soon as the last hand forms
    - Disinfect tools (fire or bleach) between plants; use clean suckers
    - Cut down and bury infected plants (single diseased stem removal)
  biological:
    - None
  chemical:
    - No chemical cure; bleach for tool disinfection only
  prevention: De-budding and tool hygiene.
  notes: Report new outbreaks to extension; community action is needed.

# ============================== MANGO ==============================
- id: mango.healthy
  crop: mango
  type: healthy
  name: {en: Healthy mango, am: ጤናማ ማንጎ}
  synonyms: [healthy, mango healthy]

- id: mango.anthracnose
  crop: mango
  type: fungal
  severity: high
  name: {en: Anthracnose}
  scientific: Colletotrichum gloeosporioides
  synonyms: [anthracnose, mango anthracnose]
  symptoms:
    - Dark irregular spots on leaves, blackened flowers, dark sunken spots on fruit that grow during ripening
  spread: Rain and dew during flowering and fruit set.
  cultural:
    - Prune for airflow; remove dead twigs and fallen leaves; harvest carefully
  biological:
    - Bacillus-based products; hot-water dip of fruit (52–55 °C for 5 minutes) after harvest
  chemical:
    - Copper, mancozeb or azoxystrobin during flowering and fruit set in wet weather
  prevention: Orchard hygiene.

- id: mango.bacterial_canker
  crop: mango
  type: bacterial
  severity: high
  name: {en: Bacterial black spot (canker)}
  scientific: Xanthomonas citri pv. mangiferaeindicae
  synonyms: [bacterial canker, mango bacterial canker, bacterial black spot]
  symptoms:
    - Raised black, angular spots with a yellow halo on leaves; cracked black spots on fruit
  spread: Wind-driven rain, wounds, infected planting material.
  cultural:
    - Windbreaks; prune and burn infected twigs; disinfect tools; clean planting material
  biological:
    - None reliable
  chemical:
    - Copper sprays after pruning and before rainy periods
  prevention: Windbreaks and hygiene.

- id: mango.powdery_mildew
  crop: mango
  type: fungal
  severity: medium
  name: {en: Powdery mildew}
  scientific: Erysiphe (Oidium) mangiferae
  synonyms: [powdery mildew, mango powdery mildew]
  symptoms:
    - White powder on young leaves, flowers and young fruit; flowers drop
  spread: Wind; cool nights and dry days during flowering.
  cultural:
    - Prune for airflow; remove infected panicles
  biological:
    - Sulphur sprays
  chemical:
    - Sulphur or triazoles at flowering
  prevention: Monitor at flowering.

- id: mango.sooty_mould
  crop: mango
  type: fungal
  severity: low
  name: {en: Sooty mould}
  scientific: Capnodium spp. (grows on honeydew from insects)
  synonyms: [sooty mould, sooty mold, mango sooty mould]
  symptoms:
    - Black, soot-like coating on leaves that can be rubbed off
  spread: Grows on honeydew from scales, mealybugs, aphids and hoppers.
  cultural:
    - Control the sap-sucking insects; prune for light and airflow
  biological:
    - Conserve ladybirds and parasitoids; soap or oil sprays against the insects
  chemical:
    - Treat the insects, not the mould
  prevention: Control honeydew-producing insects.

- id: mango.die_back
  crop: mango
  type: fungal
  severity: medium
  name: {en: Die back}
  scientific: Lasiodiplodia theobromae
  synonyms: [die back, dieback, mango die back]
  symptoms:
    - Twigs dry from the tip backwards; leaves turn brown and stay attached; gum on branches
  spread: Wounds, stress and infected wood.
  cultural:
    - Cut 10 cm below the dead part into healthy wood and burn cuttings; seal large cuts
  biological:
    - Trichoderma paste on pruning wounds
  chemical:
    - Copper paste or spray after pruning
  prevention: Good tree nutrition and pruning hygiene.

- id: mango.gall_midge
  crop: mango
  type: pest
  severity: medium
  name: {en: Gall midge damage}
  scientific: Procontarinia spp.
  synonyms: [gall midge, mango gall midge]
  symptoms:
    - Small raised warty galls on leaves; damaged flowers and young fruit
  spread: Flying midges; larvae drop to the soil to pupate.
  cultural:
    - Remove and destroy galled leaves; plough soil under trees to expose pupae
  biological:
    - Conserve parasitoids
  chemical:
    - Only on advice at flushing / flowering
  prevention: Orchard sanitation.

- id: mango.cutting_weevil
  crop: mango
  type: pest
  severity: low
  name: {en: Leaf-cutting weevil damage}
  scientific: Deporaus marginatus
  synonyms: [cutting weevil, mango cutting weevil]
  symptoms:
    - Young leaves cut off neatly near the base, as if with scissors
  spread: Adult weevils lay eggs in tender leaves.
  cultural:
    - Collect and destroy fallen cut leaves
  biological:
    - Conserve natural enemies
  chemical:
    - Only at heavy infestation during new flushes
  prevention: Sanitation during flushing.

# ============================== CITRUS ==============================
- id: citrus.healthy
  crop: citrus
  type: healthy
  name: {en: Healthy citrus, am: ጤናማ ብርቱካን}
  synonyms: [healthy, citrus healthy]

- id: citrus.greening
  crop: citrus
  type: bacterial
  severity: critical
  name: {en: Citrus greening (HLB)}
  scientific: Candidatus Liberibacter spp. (psyllids)
  synonyms: ["Orange___Haunglongbing_(Citrus_greening)", citrus greening, greening, hlb, huanglongbing]
  symptoms:
    - Blotchy yellow mottling that is not the same on both halves of the leaf; small, lopsided, bitter fruit that stays green
  spread: Psyllids (African citrus psyllid Trioza erytreae; Asian Diaphorina citri) and infected budwood.
  cultural:
    - Plant certified disease-free seedlings; remove infected trees; control psyllids area-wide
  biological:
    - Psyllid parasitoids (e.g. Tamarixia radiata)
  chemical:
    - No cure; psyllid control on extension advice
  prevention: Clean nursery stock and psyllid management.
  notes: Report suspected HLB; it is a quarantine disease in many countries.

- id: citrus.canker
  crop: citrus
  type: bacterial
  severity: high
  name: {en: Citrus canker}
  scientific: Xanthomonas citri subsp. citri
  synonyms: [canker, citrus canker, cancker, citrus cancker]
  symptoms:
    - Raised, corky brown spots with a yellow halo on leaves, twigs and fruit
  spread: Wind-driven rain, leaf miners, contaminated tools.
  cultural:
    - Windbreaks; prune infected twigs; control leaf miner; disinfect tools
  biological:
    - None reliable
  chemical:
    - Copper sprays on new flushes and young fruit
  prevention: Windbreaks and clean planting material.

- id: citrus.black_spot
  crop: citrus
  type: fungal
  severity: medium
  name: {en: Citrus black spot}
  scientific: Phyllosticta citricarpa
  synonyms: [black spot, citrus black spot]
  symptoms:
    - Hard, sunken spots with brown edges and grey centres on fruit; small spots on leaves
  spread: Leaf litter and rain during fruit development.
  cultural:
    - Remove leaf litter; prune
  biological:
    - None reliable
  chemical:
    - Copper or strobilurins during the susceptible fruit period
  prevention: Orchard hygiene.

- id: citrus.melanose
  crop: citrus
  type: fungal
  severity: low
  name: {en: Melanose}
  scientific: Diaporthe citri
  synonyms: [melanose, citrus melanose]
  symptoms:
    - Small, raised, rough dark brown dots on leaves and fruit, like sandpaper
  spread: Dead twigs in the tree; rain.
  cultural:
    - Prune out dead wood
  biological:
    - None
  chemical:
    - Copper after petal fall in wet regions
  prevention: Remove dead wood.

# ============================== APPLE ==============================
- id: apple.healthy
  crop: apple
  type: healthy
  name: {en: Healthy apple, am: ጤናማ ፖም}
  synonyms: ["Apple___healthy", apple healthy, healthy]

- id: apple.scab
  crop: apple
  type: fungal
  severity: medium
  name: {en: Apple scab}
  scientific: Venturia inaequalis
  synonyms: ["Apple___Apple_scab", apple scab, scab, apple scab leaf]
  symptoms:
    - Olive-green to black velvety spots on leaves and fruit; fruit cracks
  spread: Fallen leaves; spring rains.
  cultural:
    - Rake and destroy fallen leaves; prune for airflow; resistant varieties
  biological:
    - Sulphur, lime-sulphur
  chemical:
    - Captan, dithiocarbamates or triazoles during wet spring periods
  prevention: Leaf litter removal.

- id: apple.black_rot
  crop: apple
  type: fungal
  severity: medium
  name: {en: Black rot / frog-eye leaf spot}
  scientific: Botryosphaeria obtusa
  synonyms: ["Apple___Black_rot", apple black rot, black rot, frog eye leaf spot, frog_eye_leaf_spot]
  symptoms:
    - Leaf spots with purple edges and tan centres ("frog-eye"); fruit rot with rings; cankers on limbs
  spread: Dead wood, mummified fruit; warm, wet weather.
  cultural:
    - Remove mummified fruit and dead wood; prune out cankers
  biological:
    - None reliable
  chemical:
    - Captan or strobilurins in the season
  prevention: Sanitation.

- id: apple.cedar_rust
  crop: apple
  type: fungal
  severity: low
  name: {en: Cedar apple rust}
  scientific: Gymnosporangium spp.
  synonyms: ["Apple___Cedar_apple_rust", cedar apple rust, apple rust, rust]
  symptoms:
    - Bright yellow-orange spots on leaves, later with small tubes underneath
  spread: Alternates between juniper (cedar) and apple.
  cultural:
    - Resistant varieties; remove nearby junipers where practical
  biological:
    - None
  chemical:
    - Triazoles (e.g. myclobutanil) in spring
  prevention: Resistant varieties.

- id: apple.powdery_mildew
  crop: apple
  type: fungal
  severity: low
  name: {en: Powdery mildew}
  scientific: Podosphaera leucotricha
  synonyms: [apple powdery mildew, powdery mildew, powdery_mildew]
  symptoms:
    - White powdery coating on young leaves and shoots
  spread: Infected buds; dry warm days, humid nights.
  cultural:
    - Prune out infected shoots
  biological:
    - Sulphur
  chemical:
    - Sulphur or triazoles
  prevention: Remove infected shoots in winter pruning.

# ============================== GRAPE ==============================
- id: grape.healthy
  crop: grape
  type: healthy
  name: {en: Healthy grape, am: ጤናማ ወይን}
  synonyms: ["Grape___healthy", grape healthy, healthy]

- id: grape.black_rot
  crop: grape
  type: fungal
  severity: high
  name: {en: Black rot}
  scientific: Guignardia bidwellii
  synonyms: ["Grape___Black_rot", grape black rot, black rot, grape leaf black rot]
  symptoms:
    - Tan leaf spots with dark borders and black dots; berries shrivel into hard black mummies
  spread: Mummies and canes; warm, wet weather.
  cultural:
    - Remove mummies and infected canes; open canopy
  biological:
    - None reliable
  chemical:
    - Mancozeb, captan or triazoles from early shoot growth to berry set
  prevention: Sanitation.

- id: grape.esca
  crop: grape
  type: fungal
  severity: high
  name: {en: Esca (black measles)}
  scientific: Phaeomoniella chlamydospora and other trunk fungi
  synonyms: ["Grape___Esca_(Black_Measles)", esca, black measles]
  symptoms:
    - 'Tiger-stripe leaves: yellow or red between veins with dry centres; dark spots on berries; sudden vine collapse'
  spread: Pruning wounds.
  cultural:
    - Prune in dry weather; protect large cuts; remove dead vines
  biological:
    - Trichoderma wound protectants
  chemical:
    - No curative treatment
  prevention: Pruning-wound protection.

- id: grape.leaf_blight
  crop: grape
  type: fungal
  severity: medium
  name: {en: Leaf blight (Isariopsis leaf spot)}
  scientific: Pseudocercospora vitis
  synonyms: ["Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", isariopsis leaf spot, grape leaf blight, leaf blight]
  symptoms:
    - Dark red-brown irregular spots on leaves; early leaf drop
  spread: Warm, humid weather.
  cultural:
    - Open canopy; remove infected leaves
  biological:
    - None reliable
  chemical:
    - Mancozeb or copper if severe
  prevention: Canopy management.

# ============================== STONE FRUIT & BERRIES ==============================
- id: peach.healthy
  crop: peach
  type: healthy
  name: {en: Healthy peach, am: ጤናማ ኮክ}
  synonyms: ["Peach___healthy", peach healthy]

- id: peach.bacterial_spot
  crop: peach
  type: bacterial
  severity: medium
  name: {en: Bacterial spot}
  scientific: Xanthomonas arboricola pv. pruni
  synonyms: ["Peach___Bacterial_spot", peach bacterial spot]
  symptoms:
    - Small angular dark spots that fall out ("shot holes"); cracked spots on fruit
  spread: Wind-driven rain; sandy soils.
  cultural:
    - Resistant varieties; windbreaks; balanced nitrogen
  biological:
    - None reliable
  chemical:
    - Copper at leaf fall and early spring; oxytetracycline where registered
  prevention: Resistant varieties.

- id: cherry.healthy
  crop: cherry
  type: healthy
  name: {en: Healthy cherry}
  synonyms: ["Cherry_(including_sour)___healthy", cherry healthy]

- id: cherry.powdery_mildew
  crop: cherry
  type: fungal
  severity: low
  name: {en: Powdery mildew}
  scientific: Podosphaera clandestina
  synonyms: ["Cherry_(including_sour)___Powdery_mildew", cherry powdery mildew]
  symptoms:
    - White powdery patches on young leaves; leaves curl
  spread: Wind; warm days and humid nights.
  cultural:
    - Prune for airflow; remove root suckers
  biological:
    - Sulphur
  chemical:
    - Sulphur or triazoles
  prevention: Airflow.

- id: strawberry.healthy
  crop: strawberry
  type: healthy
  name: {en: Healthy strawberry}
  synonyms: ["Strawberry___healthy", strawberry healthy]

- id: strawberry.leaf_scorch
  crop: strawberry
  type: fungal
  severity: medium
  name: {en: Leaf scorch}
  scientific: Diplocarpon earlianum
  synonyms: ["Strawberry___Leaf_scorch", leaf scorch, strawberry leaf scorch]
  symptoms:
    - Many small purple spots that merge; leaves look burnt
  spread: Infected plants and rain splash.
  cultural:
    - Clean planting material; remove old leaves after harvest; drip irrigation
  biological:
    - None reliable
  chemical:
    - Captan or triazoles if severe
  prevention: Clean plants.

- id: blueberry.healthy
  crop: blueberry
  type: healthy
  name: {en: Healthy blueberry}
  synonyms: ["Blueberry___healthy", blueberry healthy]

- id: raspberry.healthy
  crop: raspberry
  type: healthy
  name: {en: Healthy raspberry}
  synonyms: ["Raspberry___healthy", raspberry healthy]

# ============================== COFFEE ==============================
- id: coffee.healthy
  crop: coffee
  type: healthy
  name: {en: Healthy coffee, am: ጤናማ ቡና}
  synonyms: [healthy, coffee healthy]

- id: coffee.leaf_rust
  crop: coffee
  type: fungal
  severity: high
  name: {en: Coffee leaf rust}
  scientific: Hemileia vastatrix
  synonyms: [rust, leaf rust, coffee leaf rust, rust_level_1, rust_level_2, rust_level_3, rust_level_4]
  symptoms:
    - Yellow-orange powdery spots on the underside of leaves; pale yellow patches on top
    - Heavy leaf fall and branch die-back
  spread: Wind and rain splash; warm, humid weather; more severe at lower altitudes and on heavily bearing trees.
  cultural:
    - Resistant varieties; shade management; balanced nutrition; prune for airflow
  biological:
    - None proven at field scale
  chemical:
    - Copper-based protectants before the rainy season; triazoles (e.g. cyproconazole) where registered
  prevention: Resistant varieties and good farm management.

- id: coffee.cercospora
  crop: coffee
  type: fungal
  severity: medium
  name: {en: Brown eye spot (Cercospora)}
  scientific: Cercospora coffeicola
  synonyms: [cercospora, brown eye spot, coffee cercospora, berry blotch, coffee berry blotch]
  symptoms:
    - Round brown spots with a light grey centre and yellow halo ("eye"); spots on berries
  spread: Wind and splash; stressed, under-fertilised, unshaded trees.
  cultural:
    - Shade; balanced nutrition (especially nitrogen and potassium); weed control
  biological:
    - None reliable
  chemical:
    - Copper-based fungicides
  prevention: Balanced nutrition and shade.

- id: coffee.phoma
  crop: coffee
  type: fungal
  severity: medium
  name: {en: Phoma leaf spot}
  scientific: Phoma spp.
  synonyms: [phoma, coffee phoma]
  symptoms:
    - Dark brown irregular spots, often at the leaf edges and tips; die-back of young shoots
  spread: Cold winds and high altitude; wounds.
  cultural:
    - Windbreaks; avoid cold, exposed sites
  biological:
    - None reliable
  chemical:
    - Copper or strobilurins on advice
  prevention: Windbreaks.

- id: coffee.leaf_miner
  crop: coffee
  type: pest
  severity: medium
  name: {en: Leaf miner damage}
  scientific: Leucoptera spp.
  synonyms: [leaf miner, miner, coffee leaf miner]
  symptoms:
    - Irregular brown blotch mines on the upper leaf surface; leaves drop
  spread: Moths; dry seasons.
  cultural:
    - Shade and windbreaks; avoid excessive insecticides
  biological:
    - Conserve parasitoid wasps
  chemical:
    - Only if thresholds are exceeded, using recommended insecticides
  prevention: Conserve natural enemies.

- id: coffee.red_spider_mite
  crop: coffee
  type: pest
  severity: low
  name: {en: Red spider mite damage}
  scientific: Oligonychus coffeae
  synonyms: [red spider mite, red_spider_mite, spider mite]
  symptoms:
    - Upper leaf surface turns dull bronze-brown; fine webbing
  spread: Dry weather and dust.
  cultural:
    - Shade; reduce dust; avoid broad-spectrum insecticides
  biological:
    - Predatory mites
  chemical:
    - Specific acaricides only if severe
  prevention: Protect natural enemies.

In [ ]:
%%writefile kb/fruits_round3.yaml
diseases:
# Added Oct 2026 (data round 3): mango fruit diseases, guava and papaya, sorghum smuts and grain mould,
# groundnut Alternaria. Classes are diseases, not plant parts: fruit and leaf photos of the same disease
# share one class (e.g. mango anthracnose on leaves and on fruits).
# ============================== MANGO (fruit) ==============================
- id: mango.stem_end_rot
  crop: mango
  type: fungal
  severity: medium
  name: {en: Stem-end rot}
  scientific: Lasiodiplodia theobromae (also Neofusicoccum and Phomopsis species)
  synonyms: [stem end rot, stem-end rot, stem and rot, mango stem end rot, diplodia stem end rot]
  symptoms:
    - Dark brown to black rot starting at the stalk end of ripening fruit and spreading quickly over the fruit
    - Flesh under the rot is soft and watery; dead twigs (dieback) are often present on the same tree
  spread: Lives in dead twigs and bark; infects the stalk before harvest and develops while fruit ripens, worse in warm, humid storage and when fruit is wounded.
  cultural:
    - Prune out dead twigs and branches and remove them from the orchard
    - Harvest with 5–10 cm of stalk left on, handle gently and do not drop fruit on the soil
    - Hot-water treatment after harvest (about 52–55 °C for 5 minutes), then cool, dry storage
    - Sell or eat ripe fruit quickly; separate rotting fruit
  biological:
    - Bacillus or Trichoderma-based products before harvest give partial protection
  chemical:
    - Copper sprays during fruit development; post-harvest fungicide dips only where registered for mango
  prevention: Orchard hygiene (dead wood out), careful harvest and hot-water treatment.
  notes: Often confused with anthracnose; stem-end rot starts at the stalk, anthracnose as scattered black spots.

- id: mango.scab
  crop: mango
  type: fungal
  severity: low
  name: {en: Mango scab}
  scientific: Elsinoë mangiferae
  synonyms: [scab, mango scab]
  symptoms:
    - Small grey-brown spots on young fruit that become raised, corky, cracked scabs
    - Small dark spots on young leaves, which may become distorted
  spread: Rain splash in wet weather during flowering and early fruit growth.
  cultural:
    - Prune for open canopy and good airflow; remove fallen infected leaves and fruit
  biological:
    - None reliable
  chemical:
    - Copper sprays from flowering until fruit are about half grown (same programme as for anthracnose)
  prevention: Open canopy and protection of young fruit in wet seasons.

# ============================== GUAVA ==============================
- id: guava.healthy
  crop: guava
  type: healthy
  severity: low
  name: {en: Healthy guava}
  synonyms: [healthy, healthy fruit, healthy fruits, healthy leaf, healthy leaves, guava healthy, fresh]

- id: guava.anthracnose
  crop: guava
  type: fungal
  severity: medium
  name: {en: Guava anthracnose}
  scientific: Colletotrichum gloeosporioides (C. psidii)
  synonyms: [anthracnose, guava anthracnose]
  symptoms:
    - Sunken, dark brown to black spots on fruit that join up; pink-orange spore masses in wet weather
    - Brown spots on leaves; dieback of young shoots
  spread: Rain splash from infected twigs, leaves and mummified fruit; warm, wet weather.
  cultural:
    - Remove and destroy infected and mummified fruit and dead twigs
    - Prune for airflow; harvest before full ripeness; avoid wounding fruit
  biological:
    - Bacillus or Trichoderma-based products
  chemical:
    - Copper oxychloride or mancozeb sprays at flowering and fruit set in wet seasons
  prevention: Orchard hygiene and pruning.

- id: guava.fruit_fly
  crop: guava
  type: pest
  severity: high
  name: {en: Fruit fly damage}
  scientific: Bactrocera dorsalis (oriental fruit fly) and other Bactrocera / Ceratitis species
  synonyms: [fruit fly, fruit flies, fruitfly, guava fruit fly]
  symptoms:
    - Small puncture marks on the skin, soft brown rotting patches around them
    - White maggots inside the fruit; fruit drops early
  spread: Adult flies lay eggs in ripening fruit; maggots pupate in the soil under the tree. Also attacks mango, citrus and many other fruits.
  cultural:
    - Collect all fallen and infested fruit every few days and bury it deep (over 50 cm) or seal it in plastic bags in the sun
    - Bag fruits on the tree; harvest at mature-green stage
    - Work together with neighbours: area-wide control works much better than one farm alone
  biological:
    - Male-lure traps with methyl eugenol; protein bait sprays with spinosad (spot-sprayed, not covering the tree)
    - Weaver ants and parasitic wasps (Fopius arisanus) where available
  chemical:
    - Bait sprays (protein bait plus an insecticide registered for fruit flies), applied as spots on foliage
  prevention: Orchard sanitation and traps from before fruit ripen.
  notes: Bactrocera dorsalis is one of the most damaging fruit pests in Africa; it may be a quarantine pest for export.

- id: guava.styler_end_rot
  crop: guava
  type: fungal
  severity: medium
  name: {en: Styler-end rot}
  scientific: Phomopsis psidii (and Pestalotiopsis species)
  synonyms: [styler end rot, stylar end rot, styler-end rot, style end rot]
  symptoms:
    - Brown to black rot starting at the flower (calyx) end of the fruit, opposite the stalk, spreading over the fruit
  spread: Spores from infected twigs and fruit; humid weather and wounds.
  cultural:
    - Remove infected fruit and dead twigs; avoid wounding fruit; good airflow
  biological:
    - Bacillus or Trichoderma-based products
  chemical:
    - Copper oxychloride or mancozeb sprays during fruit development in wet seasons
  prevention: Orchard hygiene.

- id: guava.scab
  crop: guava
  type: fungal
  severity: medium
  name: {en: Guava scab and canker}
  scientific: Pestalotiopsis psidii
  synonyms: [scab, canker, guava scab, guava canker, fruit canker]
  symptoms:
    - Small brown, raised, corky or cracked spots on green fruit (scab)
    - Brown, sunken, cracked lesions on leaves and fruit (canker)
  spread: Rain splash; wounds; warm, humid weather.
  cultural:
    - Remove infected fruit and leaves; prune for airflow; avoid wounding
  biological:
    - None reliable
  chemical:
    - Copper oxychloride sprays from fruit set in wet seasons
  prevention: Sanitation and protection of young fruit.

- id: guava.red_rust
  crop: guava
  type: algal
  severity: low
  name: {en: Red rust (algal leaf spot)}
  scientific: Cephaleuros virescens
  synonyms: [rust, red rust, algal leaf spot, algal spot, guava rust]
  symptoms:
    - Round, rust-red, velvety spots on leaves (and sometimes fruit and twigs)
  spread: An alga spread by rain splash; worse in humid, shaded, poorly fed trees.
  cultural:
    - Prune for light and airflow; remove heavily affected leaves; balanced fertiliser
  biological:
    - None needed in most cases
  chemical:
    - Copper oxychloride sprays if severe
  prevention: Open canopy and good tree nutrition.

# ============================== PAPAYA ==============================
- id: papaya.healthy
  crop: papaya
  type: healthy
  severity: low
  name: {en: Healthy papaya}
  synonyms: [healthy, healthy fruit, healthy leaf, healthy papaya, papaya healthy]

- id: papaya.anthracnose
  crop: papaya
  type: fungal
  severity: medium
  name: {en: Papaya anthracnose}
  scientific: Colletotrichum gloeosporioides
  synonyms: [anthracnose, papaya anthracnose]
  symptoms:
    - Round, sunken, water-soaked spots on ripening fruit that turn brown with pink-orange spores
    - Small brown spots on leaves and leaf stalks
  spread: Rain splash from infected leaves and fruit; warm, wet weather; shows mainly as fruit ripen.
  cultural:
    - Remove old leaves and infected fruit; harvest at colour break; handle fruit gently
    - Hot-water treatment after harvest (about 48 °C for 20 minutes)
  biological:
    - Bacillus-based products
  chemical:
    - Copper or mancozeb sprays at regular intervals in wet seasons
  prevention: Field hygiene and careful harvest.

- id: papaya.bacterial_spot
  crop: papaya
  type: bacterial
  severity: medium
  name: {en: Bacterial leaf spot}
  scientific: Bacteria (often Xanthomonas or Pseudomonas species)
  synonyms: [bacterial spot, bacterial leaf spot, papaya bacterial spot]
  symptoms:
    - Small, angular, water-soaked spots on leaves that turn brown, often with a yellow halo
  spread: Rain splash, wind-driven rain and tools; warm, wet weather.
  cultural:
    - Remove infected leaves; avoid overhead irrigation; do not work among wet plants
  biological:
    - Bacillus-based products
  chemical:
    - Copper sprays in wet weather
  prevention: Clean seedlings and field hygiene.

- id: papaya.leaf_curl
  crop: papaya
  type: viral
  severity: high
  name: {en: Papaya leaf curl}
  scientific: Papaya leaf curl virus (begomovirus), spread by whitefly
  synonyms: [curl, leaf curl, leaf curl virus, papaya leaf curl, leaf_curl_virus]
  symptoms:
    - Leaves curl downwards and inwards, become thick, leathery and crinkled; veins thicken
    - Plants are stunted and produce few, small fruits
  spread: Whiteflies (Bemisia tabaci) carry the virus from infected papaya, tomato, tobacco and weeds.
  cultural:
    - Remove and destroy infected plants early; control weeds; raise seedlings under insect netting
    - Do not plant papaya next to tomato, tobacco or chilli fields with whitefly problems
  biological:
    - Yellow sticky traps; neem-based sprays against whitefly
  chemical:
    - No cure for the virus; insecticides registered against whitefly reduce spread
  prevention: Clean seedlings and whitefly management.

- id: papaya.mealybug
  crop: papaya
  type: pest
  severity: high
  name: {en: Papaya mealybug}
  scientific: Paracoccus marginatus
  synonyms: [mealybug, mealybugs, papaya mealybug, mealy bug]
  symptoms:
    - White, cottony insects on leaf undersides, stems and fruit; yellowing, curling and leaf drop
    - Sticky honeydew and black sooty mould
  spread: Crawlers move by wind, people, tools and infested planting material and fruit.
  cultural:
    - Remove and destroy heavily infested leaves and fruit; do not move infested plants
  biological:
    - The parasitic wasp Acerophagus papayae (released with success in several African countries); ladybird beetles
    - Soap or neem oil sprays on small infestations
  chemical:
    - Systemic insecticides registered for mealybugs, only if biological control is not available
  prevention: Inspect new plants; protect natural enemies.
  notes: An invasive pest in East and West Africa; report new outbreaks to the extension service.

- id: papaya.mites
  crop: papaya
  type: pest
  severity: medium
  name: {en: Spider mites}
  scientific: Tetranychus species
  synonyms: [mite, mites, mite disease, spider mite, red spider mite]
  symptoms:
    - Fine yellow speckles on leaves, which turn bronze or yellow and dry; fine webbing under leaves
  spread: Hot, dry, dusty weather; wind.
  cultural:
    - Irrigate well and reduce dust; remove badly affected leaves
  biological:
    - Predatory mites; neem oil or soap sprays under the leaves
  chemical:
    - Acaricides registered for papaya, only if severe; rotate products to avoid resistance
  prevention: Avoid water stress; protect natural enemies.

- id: papaya.ringspot_virus
  crop: papaya
  type: viral
  severity: high
  name: {en: Papaya ringspot virus (mosaic)}
  scientific: Papaya ringspot virus (PRSV-P), spread by aphids
  synonyms: [ringspot, ring spot, mosaic, papaya ringspot, papaya mosaic, prsv, papaya ring spot]
  symptoms:
    - Yellow mosaic and mottling on young leaves, leaves narrow and distorted (shoe-string)
    - Dark green rings on fruit and oily streaks on stems; poor fruit set
  spread: Aphids carry the virus from infected papaya and cucurbits in seconds while probing.
  cultural:
    - Remove and destroy infected plants immediately; plant clean seedlings away from old papaya fields
    - Border crops (e.g. maize) as aphid barriers; do not plant cucurbits next to papaya
  biological:
    - Mild-strain cross protection and tolerant varieties where available
  chemical:
    - No cure; insecticides do not stop spread by aphids
  prevention: Clean seedlings, isolation and quick removal of infected plants.

# ============================== SORGHUM ==============================
- id: sorghum.loose_smut
  crop: sorghum
  type: fungal
  severity: medium
  name: {en: Loose kernel smut}
  scientific: Sporisorium cruentum
  synonyms: [loose smut, loose kernel smut, sorghum loose smut]
  symptoms:
    - Grains replaced by pointed smut sacs whose thin membrane breaks early, releasing dark spores
    - Infected plants may be shorter, with thin stalks and early heading
  spread: Seed-borne spores; infection when seed germinates.
  cultural:
    - Use clean seed from healthy fields; remove and burn smutted heads before spores are released
  biological:
    - None reliable
  chemical:
    - Seed treatment with a registered fungicide (e.g. thiram, carboxin or a triazole)
  prevention: Clean, treated seed.

- id: sorghum.covered_smut
  crop: sorghum
  type: fungal
  severity: medium
  name: {en: Covered kernel smut}
  scientific: Sporisorium sorghi
  synonyms: [covered smut, covered kernel smut, kernel smut, grain smut]
  symptoms:
    - Grains replaced by oval grey-white smut sacs that stay closed until threshing
  spread: Spores stick to seed at threshing and infect the next crop when it germinates.
  cultural:
    - Use clean seed; remove smutted heads; do not keep seed from infected fields
  biological:
    - None reliable
  chemical:
    - Seed treatment with a registered fungicide (very effective)
  prevention: Seed treatment every season.

- id: sorghum.head_smut
  crop: sorghum
  type: fungal
  severity: high
  name: {en: Head smut}
  scientific: Sporisorium reilianum
  synonyms: [head smut, sorghum head smut]
  symptoms:
    - The whole head is replaced by a large smut gall covered by a white membrane that breaks to release black spores and thread-like strands
  spread: Spores survive in soil for years and infect seedlings.
  cultural:
    - Remove and burn smutted heads before the gall opens; crop rotation; resistant varieties
  biological:
    - None reliable
  chemical:
    - Systemic seed treatment gives partial control
  prevention: Rotation, resistant varieties and removing galls.

- id: sorghum.grain_mold
  crop: sorghum
  type: fungal
  severity: high
  name: {en: Grain mould}
  scientific: Complex of Fusarium, Curvularia, Alternaria and other fungi
  synonyms: [grain mold, grain mould, grain molds, cereal grain molds, cereal grain mold, mold, mould]
  symptoms:
    - Grains discoloured pink, black, grey or white, soft and light; mould growth on the head
  spread: Rain and high humidity when grains are filling and maturing; late harvest.
  cultural:
    - Choose varieties that mature after the rains end; harvest promptly and dry grain quickly
    - Tan-plant and coloured-grain varieties with hard grain are more tolerant
  biological:
    - None reliable
  chemical:
    - Not economic in most cases
  prevention: Matching sowing date and variety to the rains; quick drying.
  notes: Mouldy grain can contain mycotoxins; do not eat or feed badly mouldy grain.

# ============================== GROUNDNUT ==============================
- id: groundnut.alternaria_leaf_spot
  crop: groundnut
  type: fungal
  severity: low
  name: {en: Alternaria leaf spot}
  scientific: Alternaria arachidis and A. tenuissima
  synonyms: [alternaria leaf spot, alternaria, groundnut alternaria leaf spot, alternaria leaf blight]
  symptoms:
    - Small brown spots with yellow margins, often at leaf tips and edges; leaf tips dry (veinal necrosis)
  spread: Spores in crop residue; warm, humid weather late in the season.
  cultural:
    - Rotation; bury or remove residue; balanced fertiliser
  biological:
    - Trichoderma-based products
  chemical:
    - Mancozeb or chlorothalonil sprays if severe (same as for leaf spots)
  prevention: Rotation and residue management.

In [ ]:
%%writefile kb/legumes_roots.yaml
diseases:
# ============================== COMMON BEAN ==============================
- id: bean.healthy
  crop: bean
  type: healthy
  name: {en: Healthy bean, am: ጤናማ ቦሎቄ}
  synonyms: [healthy, bean healthy, healthy bean]

- id: bean.angular_leaf_spot
  crop: bean
  type: fungal
  severity: high
  name: {en: Angular leaf spot}
  scientific: Pseudocercospora griseola
  synonyms: [angular leaf spot, angular_leaf_spot, als]
  symptoms:
    - Angular brown or grey spots bounded by leaf veins, often with a yellow edge
    - Spots join; leaves drop early; brown spots on pods
  spread: Seed and crop residue; rain splash and wind; warm, humid weather. One of the most damaging bean diseases in Africa.
  cultural:
    - Plant clean (certified) seed of resistant varieties
    - Rotate for at least two seasons; bury or remove residue
    - Avoid working in the field when leaves are wet
  biological:
    - Trichoderma or Bacillus seed treatment gives partial protection
  chemical:
    - Seed treatment; foliar mancozeb, chlorothalonil, or triazole sprays at first symptoms on susceptible varieties
  prevention: Clean seed of resistant varieties.

- id: bean.rust
  crop: bean
  type: fungal
  severity: medium
  name: {en: Bean rust}
  scientific: Uromyces appendiculatus
  synonyms: [bean rust, bean_rust, rust]
  symptoms:
    - Small reddish-brown powdery pustules, often with a yellow halo, mostly on the leaf underside
  spread: Wind-blown spores; long dew periods at 17–27 °C.
  cultural:
    - Resistant varieties; rotation; destroy volunteer beans and residue
  biological:
    - None proven at field scale
  chemical:
    - Triazoles (e.g. tebuconazole) or mancozeb at first pustules
  prevention: Resistant varieties.

# ============================== SOYBEAN ==============================
- id: soybean.healthy
  crop: soybean
  type: healthy
  name: {en: Healthy soybean, am: ጤናማ አኩሪ አተር}
  synonyms: [soybean healthy, "Soybean___healthy", healthy soybean]

# ============================== GROUNDNUT ==============================
- id: groundnut.healthy
  crop: groundnut
  type: healthy
  name: {en: Healthy groundnut, am: ጤናማ ለውዝ}
  synonyms: [healthy, groundnut healthy, healthy leaf, healthy_leaf]

- id: groundnut.early_leaf_spot
  crop: groundnut
  type: fungal
  severity: medium
  name: {en: Early leaf spot}
  scientific: Passalora arachidicola (Cercospora arachidicola)
  synonyms: [early leaf spot, early_leaf_spot, early leafspot]
  symptoms:
    - Round brown spots with a clear yellow halo, mainly on the upper leaf surface
  spread: Residue, rain splash; warm, humid weather.
  cultural:
    - Rotate 2–3 years away from groundnut; bury residue; resistant varieties; remove volunteers
  biological:
    - Trichoderma and neem-based sprays give partial control
  chemical:
    - Chlorothalonil, mancozeb or triazoles (e.g. tebuconazole) from 30–35 days after sowing in high-risk areas
  prevention: Rotation and residue management.

- id: groundnut.late_leaf_spot
  crop: groundnut
  type: fungal
  severity: high
  name: {en: Late leaf spot}
  scientific: Nothopassalora personata (Cercospora personata)
  synonyms: [late leaf spot, late_leaf_spot, late leafspot]
  symptoms:
    - Dark brown to black round spots, usually without a strong halo; dark tufts on the leaf underside
    - Heavy defoliation before harvest
  spread: Residue and wind; warm, wet weather late in the season.
  cultural:
    - Rotation; residue burial; resistant varieties; timely harvest
  biological:
    - Partial control with Trichoderma / neem
  chemical:
    - Chlorothalonil, mancozeb or triazoles as for early leaf spot
  prevention: Rotation and resistant varieties.

- id: groundnut.rust
  crop: groundnut
  type: fungal
  severity: medium
  name: {en: Groundnut rust}
  scientific: Puccinia arachidis
  synonyms: [groundnut rust, rust]
  symptoms:
    - Orange-brown pustules on the leaf underside that burst and release powdery spores
  spread: Wind; high humidity.
  cultural:
    - Resistant varieties; remove volunteers; early sowing
  biological:
    - None proven at field scale
  chemical:
    - Triazoles or chlorothalonil at first pustules
  prevention: Resistant varieties.

- id: groundnut.rosette
  crop: groundnut
  type: viral
  severity: high
  name: {en: Groundnut rosette}
  scientific: Groundnut rosette disease complex (spread by Aphis craccivora)
  synonyms: [rosette, groundnut rosette]
  symptoms:
    - Stunted, bushy plants with small, curled, yellow (chlorotic) or dark green mottled leaves
  spread: Aphids; worst in late-sown, sparse stands. Unique to Africa.
  cultural:
    - Resistant varieties; early sowing; dense, uniform plant stands; uproot early infected plants
  biological:
    - Conserve aphid predators (ladybirds, lacewings)
  chemical:
    - No cure; aphid control only on extension advice
  prevention: Early dense sowing with resistant varieties.

# ============================== CASSAVA ==============================
- id: cassava.healthy
  crop: cassava
  type: healthy
  name: {en: Healthy cassava, am: ጤናማ ካሳቫ}
  synonyms: [healthy, cassava healthy, healthy cassava, "4"]

- id: cassava.mosaic
  crop: cassava
  type: viral
  severity: high
  name: {en: Cassava mosaic disease (CMD)}
  scientific: Cassava mosaic begomoviruses (whitefly Bemisia tabaci)
  synonyms: [cassava mosaic disease, cmd, mosaic, cassava mosaic, "3"]
  symptoms:
    - Yellow or pale green mosaic on leaves; leaves distorted, twisted and smaller
    - Stunted plants with low root yield
  spread: Infected stem cuttings (main source) and whiteflies.
  cultural:
    - Plant cuttings only from healthy, symptom-free plants
    - Use CMD-resistant varieties
    - Uproot and destroy infected plants early (roguing)
  biological:
    - Conserve whitefly natural enemies
  chemical:
    - No chemical cure; whitefly sprays are rarely effective
  prevention: Clean planting material and resistant varieties.

- id: cassava.brown_streak
  crop: cassava
  type: viral
  severity: critical
  name: {en: Cassava brown streak disease (CBSD)}
  scientific: Cassava brown streak ipomoviruses (whitefly Bemisia tabaci)
  synonyms: [cassava brown streak disease, cbsd, brown streak, "1"]
  symptoms:
    - Yellow patches along secondary veins of older leaves; brown streaks on green stems
    - Dry, brown, corky rot inside the roots; roots unsellable
  spread: Infected cuttings and whiteflies. Spreading across East and Central Africa.
  cultural:
    - Use certified clean cuttings of tolerant varieties
    - Rogue infected plants; harvest early in affected areas
  biological:
    - None
  chemical:
    - No chemical cure
  prevention: Clean seed systems and tolerant varieties.
  notes: Report new areas of CBSD to your extension office.

- id: cassava.bacterial_blight
  crop: cassava
  type: bacterial
  severity: high
  name: {en: Cassava bacterial blight (CBB)}
  scientific: Xanthomonas axonopodis pv. manihotis (X. phaseoli pv. manihotis)
  synonyms: [cassava bacterial blight, cbb, bacterial blight, "0"]
  symptoms:
    - Angular water-soaked spots, leaf wilting and blight, gum on stems, shoot die-back
  spread: Infected cuttings, rain splash, tools; rainy season.
  cultural:
    - Clean cuttings from healthy plants; resistant varieties; rotation; disinfect tools; remove infected plants
  biological:
    - None reliable
  chemical:
    - No effective chemical control in the field
  prevention: Clean planting material.

- id: cassava.green_mottle
  crop: cassava
  type: viral
  severity: low
  name: {en: Cassava green mottle (CGM)}
  scientific: Cassava green mottle virus (also used for green mite-like mottling in datasets)
  synonyms: [cassava green mottle, cgm, green mottle, "2"]
  symptoms:
    - Small yellow-green mottled spots, leaf distortion; often confused with green mite damage
  spread: Infected cuttings.
  cultural:
    - Healthy cuttings; remove affected plants
  biological:
    - 'For green mite: predatory mites (Typhlodromalus aripo) released in many African countries'
  chemical:
    - No chemical cure for the virus
  prevention: Clean cuttings.

# ============================== POTATO ==============================
- id: potato.healthy
  crop: potato
  type: healthy
  name: {en: Healthy potato, am: ጤናማ ድንች}
  synonyms: ["Potato___healthy", potato healthy, healthy]

- id: potato.late_blight
  crop: potato
  type: oomycete
  severity: critical
  name: {en: Late blight}
  scientific: Phytophthora infestans
  synonyms: ["Potato___Late_blight", late blight, potato late blight, phytophthora, phytopthora]
  symptoms:
    - Dark, water-soaked patches on leaves that spread fast; white mould at the edge underneath in humid weather
    - Brown, rotting stems; reddish-brown dry rot in tubers
  spread: Infected tubers and volunteers; airborne spores; cool (10–24 °C), wet weather. Can destroy a field in a week.
  cultural:
    - Plant certified healthy seed tubers of resistant varieties
    - Destroy volunteer potatoes and cull piles; hill up soil over tubers
    - Remove and bury infected plants; cut haulms 2 weeks before harvest
  biological:
    - Bacillus subtilis products give weak protection; not enough alone
  chemical:
    - Protectant sprays (mancozeb, chlorothalonil, copper) before infection in wet weather
    - Systemics (metalaxyl-M + mancozeb, cymoxanil, dimethomorph) once disease is present; rotate groups
  prevention: Resistant varieties, healthy seed and a spray plan in rainy seasons.

- id: potato.early_blight
  crop: potato
  type: fungal
  severity: medium
  name: {en: Early blight}
  scientific: Alternaria solani
  synonyms: ["Potato___Early_blight", early blight, potato early blight, alternaria]
  symptoms:
    - Brown spots with concentric rings ("target board") on older leaves, often with a yellow halo
  spread: Residue and soil; warm weather with alternating wet and dry periods; stressed plants.
  cultural:
    - Rotation; balanced fertilisation; avoid drought stress; remove residue
  biological:
    - Bacillus or Trichoderma products give partial control
  chemical:
    - Mancozeb, chlorothalonil, or strobilurins / difenoconazole when spots appear on lower leaves
  prevention: Good plant nutrition and rotation.

- id: potato.bacterial_disease
  crop: potato
  type: bacterial
  severity: high
  name: {en: Bacterial disease (wilt / soft rot)}
  scientific: Ralstonia solanacearum, Pectobacterium spp. and others
  synonyms: [bacteria, bacterial wilt, potato bacteria, soft rot, blackleg]

- id: potato.fungal_disease
  crop: potato
  type: fungal
  severity: medium
  name: {en: Other fungal disease}
  scientific: Various fungi (e.g. Alternaria, Fusarium, Rhizoctonia)
  synonyms: [fungi, fungal, potato fungi]
  symptoms:
    - Leaf spots, lesions or rots that are not typical late blight
  spread: Soil, residue, seed tubers and splash; humid weather.
  cultural:
    - Rotation; certified seed; remove residue; avoid waterlogging and plant stress
  biological:
    - Trichoderma or Bacillus products give partial control
  chemical:
    - Ask your extension officer to confirm the disease before spraying; broad protectants (mancozeb, chlorothalonil) if advised
  prevention: Rotation and clean seed.
  symptoms:
    - Sudden wilting of green plants; brown ring in the cut tuber with white ooze (bacterial wilt)
    - Black, slimy stem base and soft, smelly tubers (blackleg / soft rot)
  spread: Infected seed tubers, soil, water, tools. Bacterial wilt is a major problem in African highlands.
  cultural:
    - Certified clean seed; rotate 3+ years with cereals; uproot infected plants with the soil around them
    - Clean tools; avoid irrigation water from infected fields
  biological:
    - None reliable
  chemical:
    - No effective chemical control
  prevention: Clean seed and long rotation.

- id: potato.virus
  crop: potato
  type: viral
  severity: medium
  name: {en: Potato virus diseases}
  scientific: PVY, PLRV and other viruses (aphids)
  synonyms: [virus, potato virus, potato viruses]
  symptoms:
    - Mosaic, mottling, rolled or crinkled leaves, stunted plants
  spread: Aphids and infected seed tubers.
  cultural:
    - Certified seed; rogue infected plants early; control volunteers and weeds
  biological:
    - Conserve aphid predators
  chemical:
    - No cure; mineral oil or aphicides only in seed production
  prevention: Certified seed.

- id: potato.pest_damage
  crop: potato
  type: pest
  severity: medium
  name: {en: Insect pest damage}
  scientific: e.g. potato tuber moth, aphids, leaf miners
  synonyms: [pest, potato pest, insect damage]
  symptoms:
    - Holes, mines or chewed leaves; insects or larvae visible
  spread: Varies by pest.
  cultural:
    - Hill up soil to protect tubers; remove infested plants; store tubers covered and cool
  biological:
    - Granulovirus or Bt products for tuber moth; conserve natural enemies
  chemical:
    - Only if thresholds are exceeded; use recommended insecticides
  prevention: Field hygiene and clean storage.

- id: potato.nematode
  crop: potato
  type: nematode
  severity: medium
  name: {en: Nematode damage}
  scientific: e.g. Globodera (cyst nematodes), Meloidogyne (root-knot)
  synonyms: [nematode, potato nematode]
  symptoms:
    - Patches of stunted, yellowing plants; galls on roots or tiny cysts on roots
  spread: Soil, infected tubers, tools and water.
  cultural:
    - Long rotation with non-host crops; clean seed; resistant varieties where available
  biological:
    - Organic amendments and biofumigation (brassicas) reduce populations
  chemical:
    - Nematicides only under expert guidance
  prevention: Rotation and clean seed.

In [ ]:
%%writefile kb/more_diseases.yaml
diseases:
# Added Oct 2026 for diseases present in the PlantWild / PlantSeg photo sets.
# ============================== CEREALS ==============================
- id: maize.common_smut
  crop: maize
  type: fungal
  severity: low
  name: {en: Common smut}
  scientific: Ustilago maydis
  synonyms: [corn smut, common smut, smut, maize smut]
  symptoms:
    - Large silvery-white swellings (galls) on ears, tassels, stalks or leaves that burst into black powdery spores
  spread: Spores in soil and residue; enter through wounds (hail, insects, detasseling); hot, dry weather.
  cultural:
    - Remove and destroy galls before they burst; avoid plant injuries; balanced nitrogen; rotation
  biological:
    - None reliable
  chemical:
    - Fungicides are not effective; use tolerant hybrids
  prevention: Tolerant hybrids and avoiding wounds.

- id: rice.sheath_blight
  crop: rice
  type: fungal
  severity: high
  name: {en: Sheath blight}
  scientific: Rhizoctonia solani
  synonyms: [sheath blight, rice sheath blight]
  symptoms:
    - Oval grey-green lesions with brown borders on leaf sheaths near the water line, spreading upwards; lodging
  spread: Soil-borne sclerotia float on irrigation water; dense planting, high nitrogen, warm humid weather.
  cultural:
    - Wider spacing; split nitrogen; remove weeds and stubble; drain fields periodically
  biological:
    - Trichoderma or Bacillus subtilis products
  chemical:
    - Validamycin, hexaconazole or azoxystrobin at early sheath symptoms
  prevention: Balanced nitrogen and spacing.

- id: wheat.head_scab
  crop: wheat
  type: fungal
  severity: high
  name: {en: Fusarium head blight (head scab)}
  scientific: Fusarium graminearum and related species
  synonyms: [head scab, wheat head scab, fusarium head blight, scab, fhb]
  symptoms:
    - Bleached spikelets on green heads, pink-orange mould at the base; shrivelled, chalky grains
  spread: Residue of maize and wheat; rain and high humidity at flowering. Grains can contain mycotoxins (DON).
  cultural:
    - Do not plant wheat after maize; bury residue; tolerant varieties; stagger planting dates
  biological:
    - Some Bacillus products give partial control
  chemical:
    - Triazoles (e.g. metconazole, prothioconazole, tebuconazole) at early flowering
  prevention: Rotation and residue management.
  notes: Infected grain may be unsafe to eat or feed; ask for a mycotoxin test.

- id: wheat.loose_smut
  crop: wheat
  type: fungal
  severity: medium
  name: {en: Loose smut}
  scientific: Ustilago tritici
  synonyms: [loose smut, wheat loose smut, smut]
  symptoms:
    - Heads emerge as a mass of black powdery spores instead of grain; later only the bare stalk remains
  spread: Seed-borne inside the grain; spores infect flowers of neighbouring plants.
  cultural:
    - Use certified seed; rogue smutted heads before spores spread
  biological:
    - Hot-water seed treatment
  chemical:
    - Systemic seed treatment (e.g. carboxin, tebuconazole, difenoconazole)
  prevention: Certified, treated seed.

- id: wheat.black_chaff
  crop: wheat
  type: bacterial
  severity: medium
  name: {en: Bacterial leaf streak (black chaff)}
  scientific: Xanthomonas translucens
  synonyms: [black chaff, bacterial leaf streak black chaff, bacterial leaf streak, wheat bacterial leaf streak]
  symptoms:
    - Water-soaked streaks on leaves that turn brown; dark brown-black stripes on glumes
  spread: Seed, residue, rain splash; warm, wet weather.
  cultural:
    - Clean seed; rotation; residue management; tolerant varieties
  biological:
    - None reliable
  chemical:
    - No effective chemical control
  prevention: Clean seed.

# ============================== LEGUMES ==============================
- id: bean.mosaic_virus
  crop: bean
  type: viral
  severity: high
  name: {en: Bean common mosaic}
  scientific: Bean common mosaic virus / bean common mosaic necrosis virus (aphids, seed)
  synonyms: [bean mosaic virus, mosaic virus, bean common mosaic, bcmv, mosaic]
  symptoms:
    - Light and dark green mosaic, leaf puckering and downward curling; black root (necrosis) in some varieties
  spread: Infected seed and aphids. Very common in East Africa.
  cultural:
    - Certified seed of resistant varieties; remove infected plants early; control weeds
  biological:
    - Conserve aphid predators
  chemical:
    - No cure
  prevention: Clean seed of resistant varieties.

- id: bean.halo_blight
  crop: bean
  type: bacterial
  severity: medium
  name: {en: Halo blight}
  scientific: Pseudomonas savastanoi pv. phaseolicola
  synonyms: [halo blight, bean halo blight]
  symptoms:
    - Small water-soaked spots surrounded by a wide yellow-green halo; greasy spots on pods
  spread: Seed-borne; rain splash in cool, wet weather.
  cultural:
    - Clean seed; rotation; avoid working in wet fields; bury residue
  biological:
    - None reliable
  chemical:
    - Copper sprays may slow spread
  prevention: Clean seed.

- id: soybean.rust
  crop: soybean
  type: fungal
  severity: high
  name: {en: Soybean rust}
  scientific: Phakopsora pachyrhizi
  synonyms: [soybean rust, rust, asian soybean rust]
  symptoms:
    - Small tan to reddish-brown lesions on the underside of lower leaves with tiny raised pustules; early leaf drop
  spread: Wind-blown spores; long leaf wetness. A major soybean disease in Africa.
  cultural:
    - Tolerant varieties; early planting; remove volunteer soybeans and kudzu
  biological:
    - None proven at field scale
  chemical:
    - Triazoles, strobilurins or mixtures at first symptoms, especially at flowering
  prevention: Early planting and monitoring.

- id: soybean.frogeye_leaf_spot
  crop: soybean
  type: fungal
  severity: medium
  name: {en: Frogeye leaf spot}
  scientific: Cercospora sojina
  synonyms: [frog eye leaf spot, frogeye leaf spot, soybean frog eye leaf spot]
  symptoms:
    - Round grey spots with dark reddish-brown margins on leaves
  spread: Residue and seed; warm, humid weather.
  cultural:
    - Resistant varieties; rotation; residue burial
  biological:
    - None reliable
  chemical:
    - Strobilurin or triazole fungicides if severe at early pod stage
  prevention: Resistant varieties.

- id: soybean.downy_mildew
  crop: soybean
  type: oomycete
  severity: low
  name: {en: Downy mildew}
  scientific: Peronospora manshurica
  synonyms: [downy mildew, soybean downy mildew]
  symptoms:
    - Pale yellow spots on the upper leaf surface with grey fuzzy growth underneath
  spread: Seed and residue; cool, humid weather.
  cultural:
    - Clean seed; rotation; resistant varieties
  biological:
    - None reliable
  chemical:
    - Rarely needed; seed treatment (metalaxyl) where common
  prevention: Clean seed.

- id: soybean.mosaic_virus
  crop: soybean
  type: viral
  severity: medium
  name: {en: Soybean mosaic}
  scientific: Soybean mosaic virus (aphids, seed)
  synonyms: [soybean mosaic, mosaic virus, mosaic]
  symptoms:
    - Mottled, puckered, distorted leaves; mottled seed coats
  spread: Infected seed and aphids.
  cultural:
    - Clean seed; resistant varieties; early planting
  biological:
    - Conserve aphid predators
  chemical:
    - No cure
  prevention: Virus-free seed.

- id: soybean.bacterial_blight
  crop: soybean
  type: bacterial
  severity: low
  name: {en: Bacterial blight}
  scientific: Pseudomonas savastanoi pv. glycinea
  synonyms: [bacterial blight, soybean bacterial blight]
  symptoms:
    - Small angular water-soaked spots with yellow halos that turn dark brown; ragged leaves
  spread: Seed and residue; cool, rainy weather.
  cultural:
    - Clean seed; rotation; avoid working in wet plants
  biological:
    - None reliable
  chemical:
    - Usually not economic; copper if severe
  prevention: Clean seed.

- id: soybean.brown_spot
  crop: soybean
  type: fungal
  severity: low
  name: {en: Septoria brown spot}
  scientific: Septoria glycines
  synonyms: [brown spot, soybean brown spot, septoria brown spot]
  symptoms:
    - Small irregular dark brown spots on lower leaves, which turn yellow and drop
  spread: Residue; warm, wet weather.
  cultural:
    - Rotation; residue burial
  biological:
    - None reliable
  chemical:
    - Rarely needed
  prevention: Rotation.

# ============================== VEGETABLES ==============================
- id: squash.healthy
  crop: squash
  type: healthy
  name: {en: Healthy squash / pumpkin, am: ጤናማ ዱባ}
  synonyms: [squash healthy, squash leaf, healthy]

- id: pepper.blossom_end_rot
  crop: pepper
  type: abiotic
  severity: low
  name: {en: Blossom end rot}
  scientific: Calcium deficiency in the fruit (uneven watering)
  synonyms: [blossom end rot, bell pepper blossom end rot]
  symptoms:
    - Sunken, dark, leathery patch at the bottom (blossom end) of fruits
  spread: Not infectious. Caused by uneven watering, drought, salty soil or too much nitrogen.
  cultural:
    - Water regularly and deeply; mulch; avoid excess nitrogen; lime acid soils
  biological:
    - Not applicable
  chemical:
    - Calcium foliar sprays help little; fix watering first
  prevention: Even soil moisture.

# ============================== FRUITS ==============================
- id: banana.bunchy_top
  crop: banana
  type: viral
  severity: critical
  name: {en: Banana bunchy top}
  scientific: Banana bunchy top virus (aphid Pentalonia nigronervosa)
  synonyms: [bunchy top, banana bunchy top, bbtv]
  symptoms:
    - Narrow, upright, bunched leaves at the top; dark green dots and dashes ("Morse code") on leaf veins and stalks
  spread: Banana aphids and infected suckers. Spreading in sub-Saharan Africa.
  cultural:
    - Plant virus-free (tissue-culture) plants; destroy infected mats completely, including suckers
  biological:
    - Conserve aphid enemies
  chemical:
    - No cure; kill aphids on a plant before uprooting it
  prevention: Clean planting material and quick removal.
  notes: Report new outbreaks to the plant-protection service.

- id: banana.anthracnose
  crop: banana
  type: fungal
  severity: low
  name: {en: Anthracnose}
  scientific: Colletotrichum musae
  synonyms: [anthracnose, banana anthracnose]
  symptoms:
    - Dark sunken spots on ripening fruit with orange spore masses
  spread: Wounds and rain; mainly a post-harvest problem.
  cultural:
    - Handle fruit carefully; remove dead flowers and leaves; clean packing areas
  biological:
    - Hot-water dip of hands after harvest
  chemical:
    - Post-harvest treatments only where registered
  prevention: Careful harvest and hygiene.

- id: banana.cigar_end_rot
  crop: banana
  type: fungal
  severity: low
  name: {en: Cigar end rot}
  scientific: Verticillium theobromae / Trachysphaera fructigena
  synonyms: [cigar end rot, banana cigar end rot]
  symptoms:
    - Dry, grey, ash-like rot at the tip of fingers, like the ash of a cigar
  spread: Infection through the flower; rainy weather.
  cultural:
    - Remove flower remains from young fruit; bag bunches; remove infected fingers
  biological:
    - None reliable
  chemical:
    - Rarely needed
  prevention: Remove dead flowers.

- id: banana.cordana_leaf_spot
  crop: banana
  type: fungal
  severity: low
  name: {en: Cordana leaf spot}
  scientific: Neocordana musae
  synonyms: [cordana leaf spot, banana cordana leaf spot, cordana]
  symptoms:
    - Oval pale brown spots with yellow borders, often at the leaf edge
  spread: Wind and rain; humid conditions.
  cultural:
    - Remove affected leaves; good drainage and spacing
  biological:
    - None reliable
  chemical:
    - Usually controlled together with Sigatoka programmes
  prevention: Leaf sanitation.

- id: grape.downy_mildew
  crop: grape
  type: oomycete
  severity: high
  name: {en: Downy mildew}
  scientific: Plasmopara viticola
  synonyms: [downy mildew, grape downy mildew]
  symptoms:
    - Yellow "oil spots" on the upper leaf surface with white downy growth underneath; berries shrivel
  spread: Rain splash from the soil; warm, wet weather.
  cultural:
    - Open the canopy; remove fallen leaves; avoid wetting leaves
  biological:
    - Copper is allowed in organic systems
  chemical:
    - Copper or mancozeb before rain; systemics (metalaxyl-M, dimethomorph) once infection starts
  prevention: Canopy management and timely protection.

- id: peach.leaf_curl
  crop: peach
  type: fungal
  severity: medium
  name: {en: Peach leaf curl}
  scientific: Taphrina deformans
  synonyms: [leaf curl, peach leaf curl]
  symptoms:
    - Thick, puckered, red to purple curled leaves in spring; leaves drop
  spread: Spores on bark and buds; cool, wet weather at bud break.
  cultural:
    - Remove and destroy curled leaves; plant tolerant varieties
  biological:
    - Copper or lime-sulphur in dormancy
  chemical:
    - Copper or ziram/chlorothalonil once at leaf fall or before bud swell
  prevention: Dormant spray every year.

- id: peach.brown_rot
  crop: peach
  type: fungal
  severity: high
  name: {en: Brown rot}
  scientific: Monilinia fructicola / M. laxa
  synonyms: [brown rot, peach brown rot]
  symptoms:
    - Soft brown rot on fruit covered with grey-tan spore tufts; blossom and twig blight; mummies on the tree
  spread: Mummified fruit and cankers; rain at bloom and before harvest.
  cultural:
    - Remove mummies and infected twigs; thin fruit; control fruit-damaging insects
  biological:
    - Bacillus products near harvest
  chemical:
    - Triazoles or SDHI fungicides at bloom and before harvest
  prevention: Sanitation.

- id: peach.scab
  crop: peach
  type: fungal
  severity: low
  name: {en: Peach scab}
  scientific: Venturia carpophila (Cladosporium carpophilum)
  synonyms: [peach scab, scab]
  symptoms:
    - Small olive-green to black spots on fruit, often near the stem; fruit cracks
  spread: Twig lesions; warm, humid spring.
  cultural:
    - Prune for airflow; remove infected twigs
  biological:
    - Sulphur sprays
  chemical:
    - Sulphur or captan from petal fall
  prevention: Pruning.

- id: cherry.leaf_spot
  crop: cherry
  type: fungal
  severity: medium
  name: {en: Cherry leaf spot}
  scientific: Blumeriella jaapii
  synonyms: [cherry leaf spot, leaf spot]
  symptoms:
    - Small purple spots on leaves that turn brown and may fall out; yellow leaves drop in summer
  spread: Fallen leaves; rain.
  cultural:
    - Rake and destroy fallen leaves
  biological:
    - None reliable
  chemical:
    - Captan or triazoles from petal fall in wet seasons
  prevention: Leaf litter removal.

- id: apple.mosaic_virus
  crop: apple
  type: viral
  severity: low
  name: {en: Apple mosaic}
  scientific: Apple mosaic virus (grafting)
  synonyms: [apple mosaic virus, mosaic virus, mosaic]
  symptoms:
    - Bright cream-yellow patches, bands or rings on leaves
  spread: Grafting with infected budwood; not by insects.
  cultural:
    - Use virus-tested budwood and rootstocks
  biological:
    - Not applicable
  chemical:
    - No cure
  prevention: Clean planting material.

- id: strawberry.anthracnose
  crop: strawberry
  type: fungal
  severity: medium
  name: {en: Anthracnose}
  scientific: Colletotrichum spp.
  synonyms: [anthracnose, strawberry anthracnose]
  symptoms:
    - Sunken dark lesions on fruit with orange spore masses; black spots on runners; crown rot
  spread: Infected transplants and rain splash; warm, wet weather.
  cultural:
    - Clean transplants; drip irrigation; mulch; remove infected plants
  biological:
    - Bacillus products
  chemical:
    - Captan or strobilurins in wet periods
  prevention: Clean transplants.

- id: blueberry.rust
  crop: blueberry
  type: fungal
  severity: low
  name: {en: Leaf rust}
  scientific: Thekopsora minima / Naohidemyces vaccinii
  synonyms: [blueberry rust, rust, leaf rust]
  symptoms:
    - Yellow spots on upper leaf surface with orange powdery pustules underneath; early leaf drop
  spread: Wind; alternate hosts (hemlock) in some regions.
  cultural:
    - Remove fallen leaves; prune for airflow
  biological:
    - None reliable
  chemical:
    - Triazoles after harvest if severe
  prevention: Sanitation.

In [ ]:
%%writefile kb/other.yaml
diseases:
# Photos of plants that are not one of the crops above. Lets the model say
# "I don't know this plant" instead of forcing a wrong crop. Filled by prepare.py
# from multi-crop datasets marked `other_plants: true` in pipeline/datasets.yaml.
- id: other.unknown_plant
  crop: other
  type: healthy
  severity: none
  name: {en: Plant not recognised, am: ተክሉ አልታወቀም, fr: Plante non reconnue, sw: Mmea haujatambuliwa, it: Pianta non riconosciuta}
  synonyms: []

In [ ]:
%%writefile kb/vegetables.yaml
diseases:
# ============================== TOMATO ==============================
- id: tomato.healthy
  crop: tomato
  type: healthy
  name: {en: Healthy tomato, am: ጤናማ ቲማቲም}
  synonyms: ["Tomato___healthy", tomato healthy, healthy]

- id: tomato.late_blight
  crop: tomato
  type: oomycete
  severity: critical
  name: {en: Late blight}
  scientific: Phytophthora infestans
  synonyms: ["Tomato___Late_blight", tomato late blight, late blight, tomato leaf late blight]
  symptoms:
    - Large, greasy grey-green to brown patches on leaves; white mould underneath in humid weather
    - Brown streaks on stems; firm brown blotches on fruit
  spread: Airborne spores from infected potato or tomato; cool, wet weather. Very fast.
  cultural:
    - Do not plant next to potatoes; use resistant varieties; stake and prune for airflow
    - Water at the base, not on leaves; remove and bury infected plants immediately
  biological:
    - Bacillus-based products give weak protection only
  chemical:
    - Protectants (mancozeb, chlorothalonil, copper) before rainy spells
    - Systemics (metalaxyl-M + mancozeb, cymoxanil, dimethomorph) after first symptoms; rotate groups
  prevention: Resistant varieties and a preventive spray plan in the wet season.

- id: tomato.early_blight
  crop: tomato
  type: fungal
  severity: medium
  name: {en: Early blight}
  scientific: Alternaria solani / A. linariae
  synonyms: ["Tomato___Early_blight", tomato early blight, early blight, tomato early blight leaf]
  symptoms:
    - Brown spots with concentric rings on older leaves, yellowing around spots
    - Dark sunken spots near the stem end of fruit
  spread: Soil and residue; rain splash; warm, humid weather.
  cultural:
    - Rotate 2–3 years away from tomato, potato, pepper and eggplant
    - Mulch, stake, remove lower leaves touching soil, water at the base
  biological:
    - Bacillus subtilis / Trichoderma products
  chemical:
    - Mancozeb, chlorothalonil, copper, or difenoconazole / azoxystrobin at first spots
  prevention: Rotation and mulching.

- id: tomato.bacterial_spot
  crop: tomato
  type: bacterial
  severity: medium
  name: {en: Bacterial spot}
  scientific: Xanthomonas spp.
  synonyms: ["Tomato___Bacterial_spot", tomato bacterial spot, bacterial spot, tomato leaf bacterial spot]
  symptoms:
    - Small, dark, greasy spots on leaves, sometimes with a yellow halo; scabby spots on fruit
  spread: Seed, transplants, rain splash and handling wet plants.
  cultural:
    - Clean seed and transplants; avoid overhead irrigation; do not work in wet plants; rotation
  biological:
    - Bacillus-based products and bacteriophages (where registered)
  chemical:
    - Copper products (often mixed with mancozeb) as protection; resistance to copper is common
  prevention: Clean seed and rotation.

- id: tomato.leaf_mold
  crop: tomato
  type: fungal
  severity: medium
  name: {en: Leaf mould}
  scientific: Passalora fulva (Cladosporium fulvum)
  synonyms: ["Tomato___Leaf_Mold", tomato leaf mold, leaf mold, leaf mould, tomato mold leaf]
  symptoms:
    - Pale yellow patches on the upper leaf surface; olive-green to brown velvety mould underneath
  spread: Very high humidity (over 85%), especially in greenhouses and tunnels.
  cultural:
    - Ventilate greenhouses; reduce humidity; prune for airflow; resistant varieties
  biological:
    - Bacillus-based products
  chemical:
    - Chlorothalonil, mancozeb or difenoconazole if needed
  prevention: Good ventilation.

- id: tomato.septoria_leaf_spot
  crop: tomato
  type: fungal
  severity: medium
  name: {en: Septoria leaf spot}
  scientific: Septoria lycopersici
  synonyms: ["Tomato___Septoria_leaf_spot", tomato septoria leaf spot, septoria leaf spot, septoria]
  symptoms:
    - Many small round spots with a grey-white centre and dark border on lower leaves; tiny black dots inside
  spread: Residue and weeds; rain splash; warm, wet weather.
  cultural:
    - Rotation; remove lower infected leaves; mulch; control nightshade weeds
  biological:
    - Bacillus-based products
  chemical:
    - Chlorothalonil, mancozeb or copper at first spots
  prevention: Rotation and field hygiene.

- id: tomato.spider_mites
  crop: tomato
  type: pest
  severity: medium
  name: {en: Spider mite damage}
  scientific: Tetranychus urticae / T. evansi
  synonyms: ["Tomato___Spider_mites Two-spotted_spider_mite", spider mites, two spotted spider mite, spider mite]
  symptoms:
    - Fine yellow speckling on leaves, bronzing, fine webbing on the underside
  spread: Hot, dry, dusty conditions; overuse of broad-spectrum insecticides.
  cultural:
    - Avoid drought stress; spray water under leaves; remove heavily infested leaves
  biological:
    - Predatory mites (Phytoseiulus); neem oil; insecticidal soap
  chemical:
    - Specific acaricides (e.g. abamectin, spiromesifen); rotate actives; avoid pyrethroids that worsen outbreaks
  prevention: Protect natural enemies.

- id: tomato.target_spot
  crop: tomato
  type: fungal
  severity: medium
  name: {en: Target spot}
  scientific: Corynespora cassiicola
  synonyms: ["Tomato___Target_Spot", target spot, tomato target spot]
  symptoms:
    - Brown spots with light rings that can merge; spots on fruit near the stem
  spread: Residue; warm, humid weather.
  cultural:
    - Rotation; pruning for airflow; remove residue
  biological:
    - Bacillus-based products
  chemical:
    - Chlorothalonil, mancozeb, or azoxystrobin / difenoconazole
  prevention: Rotation and airflow.

- id: tomato.yellow_leaf_curl_virus
  crop: tomato
  type: viral
  severity: high
  name: {en: Tomato yellow leaf curl virus (TYLCV)}
  scientific: Begomovirus (whitefly Bemisia tabaci)
  synonyms: ["Tomato___Tomato_Yellow_Leaf_Curl_Virus", yellow leaf curl virus, tylcv, tomato yellow leaf curl, yellow virus, leaf curl]
  symptoms:
    - Small, upward-curled leaves with yellow edges; stunted, bushy plants; few fruits
  spread: Whiteflies.
  cultural:
    - Resistant varieties; insect-proof nurseries; remove infected plants early; yellow sticky traps; control weeds
  biological:
    - Conserve whitefly enemies; neem-based products
  chemical:
    - No cure; whitefly control in nurseries and early growth only on advice
  prevention: Clean, protected seedlings and resistant varieties.

- id: tomato.mosaic_virus
  crop: tomato
  type: viral
  severity: medium
  name: {en: Tomato mosaic virus}
  scientific: Tomato mosaic virus / Tobacco mosaic virus (Tobamovirus)
  synonyms: ["Tomato___Tomato_mosaic_virus", mosaic virus, tomato mosaic virus, tomv]
  symptoms:
    - Light and dark green mosaic, fern-like or distorted leaves, uneven fruit ripening
  spread: 'Contact: hands, tools, clothing, tobacco products, seed.'
  cultural:
    - Wash hands and tools; do not smoke while handling plants; remove infected plants; resistant varieties
  biological:
    - None
  chemical:
    - No cure
  prevention: Hygiene and clean seed.

# ============================== PEPPER ==============================
- id: pepper.healthy
  crop: pepper
  type: healthy
  name: {en: Healthy pepper, am: ጤናማ ቃሪያ}
  synonyms: ["Pepper,_bell___healthy", pepper healthy, bell pepper healthy, healthy]

- id: pepper.bacterial_spot
  crop: pepper
  type: bacterial
  severity: medium
  name: {en: Bacterial spot}
  scientific: Xanthomonas spp.
  synonyms: ["Pepper,_bell___Bacterial_spot", pepper bacterial spot, bell pepper bacterial spot, bell_pepper leaf spot, bacterial spot]
  symptoms:
    - Small water-soaked spots turning brown; yellow leaves drop; raised scabby spots on fruit
  spread: Seed, transplants, rain splash.
  cultural:
    - Clean seed and transplants; rotation; avoid overhead watering
  biological:
    - Bacillus-based products
  chemical:
    - Copper (often with mancozeb) as protection
  prevention: Clean seed and rotation.

# ============================== SQUASH ==============================
- id: squash.powdery_mildew
  crop: squash
  type: fungal
  severity: medium
  name: {en: Powdery mildew}
  scientific: Podosphaera xanthii / Golovinomyces spp.
  synonyms: ["Squash___Powdery_mildew", squash powdery mildew, powdery mildew]
  symptoms:
    - White powdery patches on leaves and stems; leaves yellow and dry
  spread: Wind; warm days and cool nights; does not need rain.
  cultural:
    - Resistant varieties; good spacing; remove old infected leaves
  biological:
    - 'Sulphur, potassium bicarbonate, milk sprays (1 part milk : 9 parts water) or Bacillus products'
  chemical:
    - Sulphur, or triazoles / strobilurins at first patches; rotate groups
  prevention: Resistant varieties and spacing.

In [ ]:
%%writefile tests/test_matcher.py
"""Label-mapping tests: python3 tests/test_matcher.py"""
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "pipeline"))
from common import Matcher, load_diseases

CASES = [
 # PlantDoc folder names (crop detected from the name)
 ("Apple Scab Leaf", None, "apple.scab"), ("Apple leaf", None, "apple.healthy"), ("Apple rust leaf", None, "apple.cedar_rust"),
 ("Bell_pepper leaf spot", None, "pepper.bacterial_spot"), ("Bell_pepper leaf", None, "pepper.healthy"),
 ("Corn Gray leaf spot", None, "maize.gray_leaf_spot"), ("Corn leaf blight", None, "maize.northern_leaf_blight"),
 ("Corn rust leaf", None, "maize.common_rust"), ("Potato leaf early blight", None, "potato.early_blight"),
 ("Potato leaf late blight", None, "potato.late_blight"), ("Soyabean leaf", None, "soybean.healthy"),
 ("Squash Powdery mildew leaf", None, "squash.powdery_mildew"), ("Tomato Early blight leaf", None, "tomato.early_blight"),
 ("Tomato leaf yellow virus", None, "tomato.yellow_leaf_curl_virus"), ("Tomato leaf mosaic virus", None, "tomato.mosaic_virus"),
 ("Tomato Septoria leaf spot", None, "tomato.septoria_leaf_spot"), ("Tomato leaf bacterial spot", None, "tomato.bacterial_spot"),
 ("Tomato mold leaf", None, "tomato.leaf_mold"), ("Tomato two spotted spider mites leaf", None, "tomato.spider_mites"),
 ("grape leaf black rot", None, "grape.black_rot"), ("Strawberry leaf", None, "strawberry.healthy"), ("Peach leaf", None, "peach.healthy"),
 ("Tomato leaf", None, "tomato.healthy"), ("grape leaf", None, "grape.healthy"),
 # PlantVillage
 ("Corn_(maize)___Common_rust_", None, "maize.common_rust"), ("Pepper,_bell___healthy", None, "pepper.healthy"),
 ("Orange___Haunglongbing_(Citrus_greening)", None, "citrus.greening"),
 # single-crop datasets (crop given in datasets.yaml)
 ("0", "cassava", "cassava.bacterial_blight"), ("3", "cassava", "cassava.mosaic"), ("4", "cassava", "cassava.healthy"),
 ("angular_leaf_spot", "bean", "bean.angular_leaf_spot"), ("bean_rust", "bean", "bean.rust"), ("healthy", "bean", "bean.healthy"),
 ("bacterial_leaf_blight", "rice", "rice.bacterial_leaf_blight"), ("blast", "rice", "rice.blast"), ("normal", "rice", "rice.healthy"),
 ("dead_heart", "rice", "rice.dead_heart"), ("Anthracnose", "mango", "mango.anthracnose"), ("Sooty Mould", "mango", "mango.sooty_mould"),
 ("Die Back", "mango", "mango.die_back"), ("Bacterial Canker", "mango", "mango.bacterial_canker"),
 ("Blight", "maize", "maize.northern_leaf_blight"), ("Common_Rust", "maize", "maize.common_rust"),
 ("Gray_Leaf_Spot", "maize", "maize.gray_leaf_spot"), ("Healthy", "maize", "maize.healthy"),
 ("Phytopthora", "potato", "potato.late_blight"), ("Virus", "potato", "potato.virus"), ("Nematode", "potato", "potato.nematode"),
 ("Fungi", "potato", "potato.fungal_disease"), ("stem_rust", "wheat", "wheat.stem_rust"), ("leaf_rust", "wheat", "wheat.leaf_rust"),
 ("healthy_wheat", "wheat", "wheat.healthy"), ("miner", "coffee", "coffee.leaf_miner"), ("rust_level_2", "coffee", "coffee.leaf_rust"),
 ("red_spider_mite", "coffee", "coffee.red_spider_mite"), ("cassava mosaic disease cmd", "cassava", "cassava.mosaic"),
 ("MSV", "maize", "maize.streak_virus"), ("MLN", "maize", "maize.lethal_necrosis"),
 # numbered folders (split archives)
 ("mln4", "maize", "maize.lethal_necrosis"), ("msv2", "maize", "maize.streak_virus"), ("healthy10", "maize", "maize.healthy"),
 ("als_2", "bean", "bean.angular_leaf_spot"), ("rust_level_3", "coffee", "coffee.leaf_rust"),
 # round 3: mango fruit, guava, papaya, sorghum, groundnut, multi-crop ABCGMP (label_join)
 ("Stem End Rot", "mango", "mango.stem_end_rot"), ("Scab", "mango", "mango.scab"), ("Anthracnose", "mango", "mango.anthracnose"),
 ("Mango Healthy Fruit", None, "mango.healthy"), ("Mango Bacterial Canker", None, "mango.bacterial_canker"),
 ("Banana Cordana", None, "banana.cordana_leaf_spot"), ("Citrus Cancker Leaf", None, "citrus.canker"),
 ("Guava Red Rust", None, "guava.red_rust"), ("Papaya Ring Spot", None, "papaya.ringspot_virus"),
 ("Fruit Flies", "guava", "guava.fruit_fly"), ("Styler End Rot", "guava", "guava.styler_end_rot"), ("Canker", "guava", "guava.scab"),
 ("Rust", "guava", "guava.red_rust"), ("Healthy fruits", "guava", "guava.healthy"),
 ("Curl", "papaya", "papaya.leaf_curl"), ("Mite Disease", "papaya", "papaya.mites"), ("Mosaic", "papaya", "papaya.ringspot_virus"),
 ("Cereal Grain molds", "sorghum", "sorghum.grain_mold"), ("Covered Kernel smut", "sorghum", "sorghum.covered_smut"),
 ("loose smut", "sorghum", "sorghum.loose_smut"), ("Head Smut", "sorghum", "sorghum.head_smut"),
 ("ALTERNARIA LEAF SPOT", "groundnut", "groundnut.alternaria_leaf_spot"),
 # must NOT map without a crop
 ("healthy", None, None), ("rust", None, None), ("3", None, None),
]

if __name__ == "__main__":
    m = Matcher(load_diseases())
    bad = 0
    for raw, crop, exp in CASES:
        got = m.match(raw, crop)
        if got != exp:
            bad += 1
            print(f"FAIL {raw!r} ({crop}) -> {got}, expected {exp}")
    print(f"{len(CASES) - bad}/{len(CASES)} correct")
    sys.exit(1 if bad else 0)

## 1. Check the label mapping

In [ ]:
!python tests/test_matcher.py

## 2. Download the datasets
Failures are listed at the end and skipped; the rest still train.

In [ ]:
if MODE in ("prepare", "all"):
    skip = ("--skip " + " ".join(SKIP)) if SKIP else ""
    !python pipeline/download.py {skip} || true
    from IPython.display import Markdown, display
    display(Markdown(open("reports/download_report.md", encoding="utf-8").read()))

## 3. Clean, de-duplicate, shrink and split

In [ ]:
import shutil, tarfile
from IPython.display import Markdown, display
if MODE in ("prepare", "all"):
    !python pipeline/prepare.py --workers {WORKERS} --resize {RESIZE}
    assert (DATA / "manifest.csv").exists(), "Data preparation failed — scroll up for the error and send it to Claude"

if MODE == "prepare":
    # save the prepared photos as this notebook's output, packed in ~1 GB parts
    PREPARED.mkdir(parents=True, exist_ok=True)
    for f in ["manifest.csv", "classes.json"]:
        shutil.copy(DATA / f, PREPARED / f)
    for f in (CODE / "reports").glob("*.md"):
        shutil.copy(f, PREPARED / f.name)
    imgs = sorted((DATA / f"img{RESIZE}").iterdir())
    part, size, tf = 0, 0, None
    for p in imgs:
        if tf is None or size > 1_000_000_000:
            if tf: tf.close()
            tf = tarfile.open(PREPARED / f"images_{part:03d}.tar", "w"); part += 1; size = 0
        tf.add(p, arcname=f"img{RESIZE}/{p.name}"); size += p.stat().st_size
    if tf: tf.close()
    print(f"✓ prepared {len(imgs):,} photos in {part} parts → {PREPARED}")
    print("Run 1 is done. Now do run 2 (MODE = 'train') in a new notebook with this notebook added as input.")

if MODE == "train":
    found = [pathlib.Path(d) for d, _, fs in os.walk("/kaggle/input", followlinks=True) if "manifest.csv" in fs]
    assert found, "No prepared data found. Add Input → Your Work → the notebook from run 1, then run again."
    src = found[0]
    print("prepared data:", src)
    for f in ["manifest.csv", "classes.json"]:
        shutil.copy(src / f, DATA / f)
    for f in src.glob("*.md"):
        shutil.copy(f, CODE / "reports" / f.name)
    for t in sorted(src.glob("images_*.tar")):
        with tarfile.open(t) as tf:
            tf.extractall(DATA, filter="data")
    print("✓ unpacked", sum(1 for _ in (DATA / f"img{RESIZE}").iterdir()), "photos")

display(Markdown(open("reports/prepare_report.md", encoding="utf-8").read()))

## Smoke test (≈5 minutes)
Runs every later step on a tiny scale, so a problem shows up now instead of after hours of training.

In [ ]:
if TRAIN:
    S = RUNS / "smoke"
    !python pipeline/train.py --model {STUDENT} --img {STUDENT_IMG} --epochs 1 --epoch-size 640 --bs 32 --max-steps 20 --workers {WORKERS} --out {S}
    !python pipeline/train.py --model {STUDENT} --img {STUDENT_IMG} --epochs 1 --epoch-size 640 --bs 32 --max-steps 10 --workers {WORKERS} --teacher {S}/best.pt --out {S}/kd
    !python pipeline/evaluate.py {S}/best.pt --max-images 300 --workers {WORKERS} --out {S}/eval.json
    !python pipeline/export.py {S}/best.pt --quantize --calib 20 --out {S}/model
    !python pipeline/evaluate.py {S}/model/model.int8.onnx --max-images 300 --workers {WORKERS} --out {S}/eval_onnx.json
    assert (S / "model/model.int8.onnx").exists() and (S / "eval_onnx.json").exists(), "Smoke test failed — scroll up for the error"
    print("✓ smoke test passed: training, distillation, evaluation and export all work")

## 4. Train the teacher (large, accurate)

In [ ]:
if TRAIN:
    # fit both trainings into the time that is left (keep 50 min for evaluation + export at the end)
    left = minutes_left() - 50
    teacher_min = int(min(TEACHER_MIN, left * 0.45))
    print(f"{minutes_left():.0f} min left in the session → teacher gets {teacher_min} min")
    if USE_TEACHER and teacher_min >= 40:
        !python pipeline/train.py --model {TEACHER} --img {TEACHER_IMG} --epochs 40 --epoch-size {EPOCH_SIZE} --bs 64 --lr 4e-4 --workers {WORKERS} --time-limit {teacher_min} --out {RUNS}/teacher

## 5. Train the phone model (distilled from the teacher)

In [ ]:
if TRAIN:
    teacher_arg = f"--teacher {RUNS}/teacher/best.pt" if USE_TEACHER and (RUNS / "teacher/best.pt").exists() else ""
    student_min = int(min(STUDENT_MIN, minutes_left() - 50))
    print(f"{minutes_left():.0f} min left in the session → phone model gets {student_min} min")
    !python pipeline/train.py --model {STUDENT} --img {STUDENT_IMG} --epochs 60 --epoch-size {EPOCH_SIZE} --bs 96 --lr 1e-3 --workers {WORKERS} --time-limit {student_min} {teacher_arg} --out {RUNS}/student

## 6. Test on photos the models never saw

In [ ]:
if TRAIN:
    if (RUNS / "teacher/best.pt").exists():
        !python pipeline/evaluate.py {RUNS}/teacher/best.pt --workers {WORKERS} --out reports/eval_teacher.json
    !python pipeline/evaluate.py {RUNS}/student/best.pt --workers {WORKERS} --out reports/eval_student.json

## 7. Export for the app and check the exported model

In [ ]:
if TRAIN:
    !python pipeline/build_kb.py
    !python pipeline/export.py {RUNS}/student/best.pt --quantize
    !python pipeline/evaluate.py app/model/model.onnx --workers {WORKERS} --out reports/eval_onnx_fp32.json
    !python pipeline/evaluate.py app/model/model.int8.onnx --workers {WORKERS} --out reports/eval_onnx_int8.json

## 8. Package the result → Output tab → `sebel_hakim_model.zip`

In [ ]:
if TRAIN:
    import json, shutil, zipfile
    fp32 = json.load(open("reports/eval_onnx_fp32.json")); int8 = json.load(open("reports/eval_onnx_int8.json"))
    drop = int8["overall"]["acc"] < fp32["overall"]["acc"] - 0.01
    labels = json.load(open("app/model/labels.json"))
    if drop:                                    # int8 lost too much accuracy → ship the full-precision model
        labels["files"].pop("int8", None); (CODE / "app/model/model.int8.onnx").unlink(missing_ok=True)
        json.dump(labels, open("app/model/labels.json", "w"), indent=1)
    print(f"fp32 acc {fp32['overall']['acc']:.3f} | int8 acc {int8['overall']['acc']:.3f} → shipping {'fp32' if drop else 'int8'}")
    shutil.copy(RUNS / "student/best.pt", CODE / "student_best.pt")
    with zipfile.ZipFile("/kaggle/working/sebel_hakim_model.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for p in list((CODE / "app/model").glob("*")) + list((CODE / "reports").glob("*")):
            z.write(p, p.relative_to(CODE))
        for name in ["log.json"]:
            for run in ["teacher", "student"]:
                if (RUNS / run / name).exists():
                    z.write(RUNS / run / name, f"runs/{run}/{name}")
    print("✓ /kaggle/working/sebel_hakim_model.zip", round(pathlib.Path("/kaggle/working/sebel_hakim_model.zip").stat().st_size / 1e6, 1), "MB")
    print(json.dumps({k: fp32[k] for k in ["overall", "field_only", "lab_only", "crop_correct", "not_sure_rate"]}, indent=1))